# Appliance Energy Forecasting — complete analysis

**MSc Data Science — time-series forecasting assignment**

Single self-contained notebook. Runs top to bottom in Google Colab or locally
with no other project files required.

**Dataset:** UCI *Appliances Energy Prediction* (Candanedo et al., 2017) —
19,735 observations at 10-minute resolution, aggregated to hourly.

**Target:** `Appliances` (Wh) · **Horizon:** 24 hours · **Test:** final 14 days

### Contents

| Section | Assignment part |
|---|---|
| 1. Setup and configuration | — |
| 2. Data loading, quality checks, aggregation | Part 1 |
| 3. Exploratory analysis | Part 1 |
| 4. Stationarity and seasonality | Part 2 |
| 5. Forecasting problem definition | Part 3 |
| 6. Benchmark models | Part 4 |
| 7. SARIMA / SARIMAX with AIC grid search | Part 5 |
| 8. Feature engineering and ML model | Parts 6–7 |
| 9. Foundation model (Chronos) | Part 8 |
| 10. Evaluation and comparison | Parts 9–10 |
| 11. Analysis outputs and leakage audit | Parts 11–12 |

### Runtime

About **12 minutes** with `RUN_FULL_GRID = False` (the SARIMAX order is taken
from the completed search), or about **45 minutes** with `RUN_FULL_GRID = True`,
which refits all 168 candidate models from scratch.

> **On Colab, the foundation model works.** Colab has network access to the
> Hugging Face Hub, so Chronos runs and Question 4 becomes answerable.

---
## 1. Setup and configuration

### 1.1 Install dependencies

In [ ]:
# Colab already has numpy/pandas/scipy/matplotlib/sklearn/statsmodels.
# Chronos is optional: without it the notebook still completes and honestly
# records the foundation model as NOT EXECUTED.

INSTALL_CHRONOS = True   # set False to skip the ~2 GB torch download

import importlib.util, subprocess, sys

if INSTALL_CHRONOS and importlib.util.find_spec("chronos") is None:
    print("Installing chronos-forecasting (this takes a few minutes)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "chronos-forecasting"], check=False)

for package in ["numpy", "pandas", "scipy", "sklearn", "statsmodels", "matplotlib"]:
    spec = importlib.util.find_spec(package)
    print(f"{package:14s} {'OK' if spec else 'MISSING'}")
print(f"{'chronos':14s} {'OK' if importlib.util.find_spec('chronos') else 'not installed'}")

### 1.2 Imports

In [ ]:
import gc, hashlib, itertools, json, platform, signal, sys, time, urllib.error, urllib.request, warnings
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

warnings.filterwarnings("ignore")
%matplotlib inline
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)
print("Imports OK")

### 1.3 Configuration

Every tunable constant lives here, so the experiment is reproducible and there
are no magic numbers buried in the modelling code.

In [ ]:
"""
Central configuration for the appliance energy forecasting project.

Every tunable constant lives here so that experiments are reproducible and
so that no "magic numbers" are buried inside the modelling code.
"""


from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()   # notebooks have no __file__

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
FORECAST_DIR = OUTPUT_DIR / "forecasts"
METRICS_DIR = OUTPUT_DIR / "metrics"
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
MODEL_DIR = OUTPUT_DIR / "models"

ALL_DIRS = [
    RAW_DIR,
    PROCESSED_DIR,
    FIGURE_DIR,
    FORECAST_DIR,
    METRICS_DIR,
    ANALYSIS_DIR,
    MODEL_DIR,
]


def ensure_dirs() -> None:
    """Create every output directory used by the pipeline."""
    for path in ALL_DIRS:
        path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Data source
# ------------------------------------------------------------------

# Canonical source (UCI Machine Learning Repository).
RAW_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00374/energydata_complete.csv"
)

# Mirror maintained by the dataset authors (Candanedo et al., 2017).
# Used only if the canonical URL is unreachable (e.g. firewalled CI runners).
MIRROR_URL = (
    "https://raw.githubusercontent.com/LuisM78/"
    "Appliances-energy-prediction-data/master/energydata_complete.csv"
)

RAW_FILENAME = "energydata_complete.csv"
PROCESSED_FILENAME = "appliance_hourly.csv"

# Expected shape of the raw file. Used as a data-integrity assertion so that a
# silently truncated or altered download is caught immediately.
EXPECTED_RAW_ROWS = 19_735
EXPECTED_RAW_FREQ_MINUTES = 10

# ------------------------------------------------------------------
# Target and frequency
# ------------------------------------------------------------------

TARGET = "Appliances"
TARGET_UNITS = "Wh (mean per 10-min interval within the hour)"

FREQ = "h"
OBS_PER_HOUR_RAW = 6  # 60 / 10 minutes

DAILY_PERIOD = 24  # observations per day at hourly frequency
WEEKLY_PERIOD = 168  # observations per week at hourly frequency
SEASONAL_PERIOD = DAILY_PERIOD  # primary seasonal period used by SARIMAX

# ------------------------------------------------------------------
# Forecasting experiment design
# ------------------------------------------------------------------

# The assignment requires (a) a 24-hour forecast horizon and (b) the final
# 14 days as the test period. These are reconciled with a rolling-origin
# evaluation: 14 consecutive origins, each producing a 24-step-ahead forecast.
HORIZON = 24
TEST_DAYS = 14
TEST_STEPS = TEST_DAYS * 24  # 336 hourly observations
N_ORIGINS = TEST_STEPS // HORIZON  # 14 rolling origins

PRIMARY_METRIC = "MASE"  # scale-free and comparable to the seasonal naive
SECONDARY_METRIC = "RMSE"

# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------

RANDOM_STATE = 0

# ------------------------------------------------------------------
# Variable groups
# ------------------------------------------------------------------

# Random variates included in the raw file purely as noise controls by the
# original authors. They carry no information by construction and are dropped.
NOISE_COLS = ["rv1", "rv2"]

# Indoor sensors (9 rooms x temperature/humidity).
INDOOR_TEMP_COLS = [f"T{i}" for i in range(1, 10)]
INDOOR_RH_COLS = [f"RH_{i}" for i in range(1, 10)]
INDOOR_COLS = INDOOR_TEMP_COLS + INDOOR_RH_COLS

# Outdoor weather from the nearest Chievres airport station.
WEATHER_COLS = [
    "T_out",
    "RH_out",
    "Windspeed",
    "Visibility",
    "Tdewpoint",
    "Press_mm_hg",
]

# A second metered energy channel (lighting circuits). It is NOT known ahead of
# time, so it may only ever be used at a lag.
OTHER_ENERGY_COLS = ["lights"]

# Candidate exogenous regressors for SARIMAX (subject to selection in
# src/sarimax_model.py -- these are candidates, not the final set).
SARIMAX_CANDIDATE_EXOG = ["T_out", "RH_out", "Windspeed", "Visibility", "Tdewpoint"]

# ------------------------------------------------------------------
# Feature engineering
# ------------------------------------------------------------------

TARGET_LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
ROLLING_WINDOWS = [3, 6, 12, 24, 168]
EXOG_LAGS = [24, 48]  # only lags >= HORIZON are observable at the forecast origin

# ------------------------------------------------------------------
# SARIMAX grid search
# ------------------------------------------------------------------
# The assignment requires a search over p in [0, 6], d in [0, 2], q in [0, 6]
# with appropriate seasonal terms at s = 24.
#
# COMPUTATIONAL NOTE (documented, not hidden):
# A fully crossed search of 7*3*7 non-seasonal x 2*2*2 seasonal orders = 1176
# SARIMAX fits at s=24. Measured single-fit cost on this machine ranges from
# ~2 s to ~200 s, i.e. a fully crossed search is several days of compute on one
# core. The search is therefore staged, which preserves the *entire* required
# (p, d, q) space rather than shrinking it:
#
#   Stage 1: the complete required grid p in [0,6] x d in [0,2] x q in [0,6]
#            (147 models) is fitted with the seasonal structure held at
#            (0, D*, 0, 24), where D* is chosen from seasonal-strength evidence.
#            Ranked by AIC.
#   Stage 2: the TOP_K non-seasonal orders from stage 1 are crossed with the
#            full seasonal grid P, D, Q in {0, 1} at s = 24.
#
# AIC is comparable across differencing orders here because statsmodels'
# state-space SARIMAX (simple_differencing=False, the default) keeps the number
# of observations fixed at nobs regardless of d and D. This is verified in
# src/sarimax_model.py.
SEARCH_P = list(range(0, 7))
SEARCH_D = list(range(0, 3))
SEARCH_Q = list(range(0, 7))
SEARCH_SEASONAL_P = [0, 1]
SEARCH_SEASONAL_D = [0, 1]
SEARCH_SEASONAL_Q = [0, 1]

GRID_TOP_K = 3  # non-seasonal orders promoted from stage 1 to stage 2
GRID_MAXITER = 300  # optimiser iterations during search (final model is refitted fully)
GRID_FIT_TIMEOUT_S = 150  # per-model wall-clock guard (seconds); exceeded fits are recorded as "timeout"
SARIMAX_TREND = "c"

# ------------------------------------------------------------------
# Machine learning model
# ------------------------------------------------------------------

ML_CV_SPLITS = 4  # TimeSeriesSplit folds for hyper-parameter selection
ML_PARAM_GRID = [
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 31, "min_samples_leaf": 20},
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 63, "min_samples_leaf": 20},
    {"learning_rate": 0.10, "max_iter": 300, "max_leaf_nodes": 31, "min_samples_leaf": 40},
    {"learning_rate": 0.03, "max_iter": 600, "max_leaf_nodes": 31, "min_samples_leaf": 20},
]

# ------------------------------------------------------------------
# Foundation model
# ------------------------------------------------------------------

CHRONOS_MODEL_ID = "amazon/chronos-bolt-base"
CHRONOS_CONTEXT_LENGTH = 512  # hours of context fed to the model at each origin

# ------------------------------------------------------------------
# Plotting
# ------------------------------------------------------------------

FIG_DPI = 200
FIG_FORMAT = "png"


In [ ]:
# Rebuild a `cfg` namespace so the module code below works unchanged.
import types
cfg = types.SimpleNamespace(**{k: v for k, v in globals().items() if k.isupper()})
cfg.ensure_dirs = ensure_dirs
cfg.ensure_dirs()

print("Target:      ", cfg.TARGET)
print("Horizon:     ", cfg.HORIZON, "hours")
print("Test period: ", cfg.TEST_DAYS, "days =", cfg.TEST_STEPS, "observations")
print("Origins:     ", cfg.N_ORIGINS)
print("Outputs ->   ", cfg.OUTPUT_DIR)

---
## 2. Data loading, quality checks and hourly aggregation

**Assignment Part 1.** Download, parse, assess quality, aggregate to hourly.

The UCI URL is tried first, with an automatic fallback to the mirror published by the dataset authors if that host is unreachable.

In [ ]:
"""
Data retrieval, quality assessment and hourly aggregation.

Covers assignment Part 1, steps 1-11.

Design notes
------------
* The raw file is cached on disk. Re-runs are therefore offline and
  byte-identical, which matters for reproducibility.
* No imputation decision is made before the quality report is produced: the
  report drives the decision rather than the other way round.
* Aggregation happens *before* the train/test split, but it uses only
  within-hour information (six 10-minute readings belonging to the same
  timestamp bucket). No information crosses an hour boundary, so hourly
  aggregation cannot leak future values into past rows. This is asserted in
  the leakage audit.
"""


import hashlib
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd



# ------------------------------------------------------------------
# Download / cache
# ------------------------------------------------------------------


def download_raw(force: bool = False, timeout: int = 120) -> pd.DataFrame:
    """
    Fetch ``energydata_complete.csv`` and cache it under ``data/raw/``.

    The canonical UCI URL is tried first. If it is unreachable (some sandboxed
    or firewalled environments block ``archive.ics.uci.edu``), the mirror
    published by the dataset authors is used instead. Both serve the identical
    file; the row count is asserted afterwards.

    Parameters
    ----------
    force : bool
        Re-download even if a cached copy exists.
    timeout : int
        Per-request timeout in seconds.

    Returns
    -------
    pandas.DataFrame
        The raw, unparsed table exactly as distributed.
    """
    cfg.ensure_dirs()
    target = cfg.RAW_DIR / cfg.RAW_FILENAME

    if target.exists() and not force:
        print(f"[data] Using cached raw file: {target}")
    else:
        last_error: Exception | None = None
        for label, url in (("UCI", cfg.RAW_URL), ("author mirror", cfg.MIRROR_URL)):
            try:
                print(f"[data] Downloading from {label}: {url}")
                with urllib.request.urlopen(url, timeout=timeout) as response:
                    payload = response.read()
                target.write_bytes(payload)
                print(f"[data] Saved {len(payload):,} bytes to {target}")
                last_error = None
                break
            except (urllib.error.URLError, TimeoutError, OSError) as exc:
                print(f"[data] {label} unreachable: {exc}")
                last_error = exc
        if last_error is not None:
            raise RuntimeError(
                "Could not download the dataset from either the UCI URL or the "
                "author mirror. Download it manually and place it at "
                f"{target}."
            ) from last_error

    digest = hashlib.md5(target.read_bytes()).hexdigest()
    print(f"[data] MD5 of raw file: {digest}")

    raw = pd.read_csv(target)
    return raw


# ------------------------------------------------------------------
# Quality report
# ------------------------------------------------------------------


@dataclass
class QualityReport:
    """Structured record of the data quality checks (Part 1, step 5)."""

    n_rows: int
    n_cols: int
    start: str
    end: str
    missing_total: int
    missing_by_col: dict[str, int]
    duplicated_timestamps: int
    duplicated_rows: int
    dtypes: dict[str, str]
    inferred_step_minutes: float
    irregular_steps: int
    expected_regular_rows: int
    missing_timestamps: int
    target_min: float
    target_max: float
    target_zero_count: int
    target_negative_count: int
    target_outlier_count_iqr: int
    target_outlier_share_iqr: float
    notes: list[str] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {k: getattr(self, k) for k in self.__dataclass_fields__}


def assess_quality(df: pd.DataFrame, target: str = cfg.TARGET) -> QualityReport:
    """
    Run the Part 1 data quality checks on the *raw* 10-minute table.

    Checks: missing values, duplicated timestamps, duplicated rows, dtypes,
    sampling regularity, and anomalous target values.
    """
    step = df.index.to_series().diff().dropna()
    step_minutes = step.dt.total_seconds() / 60.0
    modal_step = float(step_minutes.mode().iloc[0]) if len(step_minutes) else np.nan
    irregular = int((step_minutes != modal_step).sum())

    expected = pd.date_range(df.index.min(), df.index.max(), freq=f"{int(modal_step)}min")

    y = df[target].astype(float)
    q1, q3 = y.quantile(0.25), y.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    lower = q1 - 1.5 * iqr
    outliers = int(((y > upper) | (y < lower)).sum())

    notes: list[str] = []
    if irregular == 0:
        notes.append(
            f"Sampling is perfectly regular at {modal_step:.0f}-minute intervals; "
            "no gaps to impute at the raw resolution."
        )
    if outliers:
        notes.append(
            f"{outliers} target values lie outside the 1.5*IQR fence (> {upper:.1f} Wh). "
            "These are high-consumption spikes, i.e. genuine appliance events, "
            "not sensor faults; they are retained."
        )
    if int((y <= 0).sum()) == 0:
        notes.append(
            "The target is strictly positive (min "
            f"{y.min():.0f} Wh), so MAPE is computable, but the heavy right skew "
            "still makes it unstable. MASE is used as the primary metric."
        )

    return QualityReport(
        n_rows=int(df.shape[0]),
        n_cols=int(df.shape[1]),
        start=str(df.index.min()),
        end=str(df.index.max()),
        missing_total=int(df.isna().sum().sum()),
        missing_by_col={c: int(v) for c, v in df.isna().sum().items() if v > 0},
        duplicated_timestamps=int(df.index.duplicated().sum()),
        duplicated_rows=int(df.duplicated().sum()),
        dtypes={c: str(t) for c, t in df.dtypes.items()},
        inferred_step_minutes=modal_step,
        irregular_steps=irregular,
        expected_regular_rows=int(len(expected)),
        missing_timestamps=int(len(expected.difference(df.index))),
        target_min=float(y.min()),
        target_max=float(y.max()),
        target_zero_count=int((y == 0).sum()),
        target_negative_count=int((y < 0).sum()),
        target_outlier_count_iqr=outliers,
        target_outlier_share_iqr=float(outliers / len(y)),
        notes=notes,
    )


# ------------------------------------------------------------------
# Parsing and aggregation
# ------------------------------------------------------------------


def parse_raw(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Parse ``date`` as datetime, index by it, sort chronologically and coerce
    every remaining column to numeric (Part 1, steps 2-4).
    """
    df = raw.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df.index.name = "date"
    return df


def resample_hourly(
    df: pd.DataFrame,
    target: str = cfg.TARGET,
    min_obs_per_hour: int = cfg.OBS_PER_HOUR_RAW,
    drop_noise_cols: bool = True,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Aggregate 10-minute observations to hourly observations (Part 1, steps 7-10).

    Aggregation method and justification
    ------------------------------------
    The **hourly mean** is used for every column.

    * The indoor/outdoor sensor channels (temperatures, humidities, pressure,
      wind, visibility) are *instantaneous state* measurements. Summing them is
      physically meaningless; the mean is the only defensible aggregate.
    * ``Appliances`` is energy use in Wh recorded per 10-minute interval. Both
      the sum (= total energy consumed in the hour) and the mean (= average
      10-minute consumption during the hour) are defensible. The mean is chosen
      for three reasons: (i) it keeps a single, consistent aggregation rule
      across all columns; (ii) it is robust to hours that contain fewer than six
      readings, whereas a sum would be biased downwards for such hours;
      (iii) sum = 6 x mean exactly, so the choice is a fixed linear rescaling.
      It leaves every scale-free metric (MASE, sMAPE) and every model ranking
      unchanged, and simply rescales MAE/RMSE/Bias by a factor of 6.

    Partial hours
    -------------
    Hours containing fewer than ``min_obs_per_hour`` raw readings are dropped
    rather than imputed, because a partial hour is a *known-incomplete*
    aggregate rather than a missing value. In this dataset only the final
    timestamp forms a partial hour.

    Returns
    -------
    (hourly, info)
        ``hourly`` is the aggregated frame; ``info`` records the decisions made
        so they can be written into the analysis summary.
    """
    frame = df.copy()

    dropped_noise: list[str] = []
    if drop_noise_cols:
        dropped_noise = [c for c in cfg.NOISE_COLS if c in frame.columns]
        frame = frame.drop(columns=dropped_noise)

    counts = frame[target].resample(cfg.FREQ).count()
    hourly = frame.resample(cfg.FREQ).mean()

    incomplete = counts[counts < min_obs_per_hour]
    hourly = hourly.loc[counts[counts >= min_obs_per_hour].index]

    missing_after = int(hourly.isna().sum().sum())

    # Any residual gap is filled by time interpolation. This is safe with
    # respect to leakage only if it does not run across the train/test boundary
    # -- see `check_interpolation_safety`. On this dataset the count is zero.
    n_interpolated = 0
    if missing_after > 0:
        before = hourly.isna().sum().sum()
        hourly = hourly.interpolate(method="time", limit_direction="forward")
        hourly = hourly.dropna()
        n_interpolated = int(before - hourly.isna().sum().sum())

    expected_hours = pd.date_range(hourly.index.min(), hourly.index.max(), freq=cfg.FREQ)
    gaps = expected_hours.difference(hourly.index)

    info = {
        "aggregation_method": "mean",
        "aggregation_justification": (
            "Sensor channels are instantaneous states (sum is meaningless); the "
            "target is per-interval Wh where sum = 6 x mean, a fixed linear "
            "rescaling that leaves MASE/sMAPE and all model rankings unchanged."
        ),
        "dropped_noise_columns": dropped_noise,
        "n_hours": int(len(hourly)),
        "n_partial_hours_dropped": int(len(incomplete)),
        "partial_hours_dropped": [str(t) for t in incomplete.index],
        "missing_cells_after_resampling": missing_after,
        "n_values_interpolated": n_interpolated,
        "remaining_hourly_gaps": int(len(gaps)),
        "start": str(hourly.index.min()),
        "end": str(hourly.index.max()),
    }
    return hourly, info


def check_interpolation_safety(info: dict[str, Any]) -> str:
    """
    Report whether interpolation could have moved information across the
    train/test boundary (Part 1, step 10 / leakage audit).
    """
    if info["n_values_interpolated"] == 0:
        return (
            "No interpolation was applied (zero missing cells after "
            "aggregation), so no imputation-induced leakage is possible."
        )
    return (
        f"{info['n_values_interpolated']} cells were time-interpolated. "
        "Time interpolation is bidirectional and can therefore blend a future "
        "observation into a past one. Verify that no interpolated timestamp "
        "falls within HORIZON hours of the train/test boundary."
    )


# ------------------------------------------------------------------
# Split
# ------------------------------------------------------------------


def train_test_split_chronological(
    series_or_frame: pd.Series | pd.DataFrame,
    test_steps: int = cfg.TEST_STEPS,
) -> tuple[Any, Any]:
    """
    Strictly chronological hold-out split (Part 3).

    The last ``test_steps`` observations form the test period. There is no
    shuffling anywhere in this project.
    """
    if len(series_or_frame) <= test_steps:
        raise ValueError(
            f"Series of length {len(series_or_frame)} is too short for a "
            f"{test_steps}-step test set."
        )
    return series_or_frame.iloc[:-test_steps], series_or_frame.iloc[-test_steps:]


def load_processed() -> pd.DataFrame:
    """Load the cached hourly dataset written by :func:`build_dataset`."""
    path = cfg.PROCESSED_DIR / cfg.PROCESSED_FILENAME
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run `python scripts/run_pipeline.py` first."
        )
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "date"
    return df


def build_dataset(force_download: bool = False) -> tuple[pd.DataFrame, QualityReport, dict[str, Any]]:
    """
    End-to-end Part 1 routine: download -> parse -> assess -> aggregate -> save.
    """
    cfg.ensure_dirs()

    raw = download_raw(force=force_download)
    parsed = parse_raw(raw)

    if len(parsed) != cfg.EXPECTED_RAW_ROWS:
        print(
            f"[data] WARNING: expected {cfg.EXPECTED_RAW_ROWS} raw rows, "
            f"got {len(parsed)}. The source file may have changed."
        )

    report = assess_quality(parsed)
    print(
        f"[data] Raw: {report.n_rows} rows x {report.n_cols} cols, "
        f"{report.start} -> {report.end}, "
        f"step={report.inferred_step_minutes:.0f} min, "
        f"missing={report.missing_total}, dup_ts={report.duplicated_timestamps}"
    )

    hourly, info = resample_hourly(parsed)
    print(
        f"[data] Hourly: {info['n_hours']} rows, "
        f"{info['n_partial_hours_dropped']} partial hour(s) dropped, "
        f"{info['n_values_interpolated']} value(s) interpolated"
    )

    out_path = cfg.PROCESSED_DIR / cfg.PROCESSED_FILENAME
    hourly.to_csv(out_path)
    print(f"[data] Saved processed hourly dataset -> {out_path}")

    return hourly, report, info


---
## 3. Exploratory analysis (functions)

**Assignment Part 1.** Descriptive statistics and pattern quantification.

In [ ]:
"""
Exploratory data analysis computations (assignment Part 1, steps 12-13).

Numbers live here; figures live in :mod:`src.plotting`. Keeping them apart
means every claim in the report can be traced to a value in the analysis
summary rather than read off a chart by eye.
"""


from typing import Any

import numpy as np
import pandas as pd



def summary_statistics(y: pd.Series) -> dict[str, Any]:
    """Descriptive statistics for the target, including shape diagnostics."""
    values = y.dropna()
    return {
        "n": int(len(values)),
        "start": str(values.index.min()),
        "end": str(values.index.max()),
        "mean": round(float(values.mean()), 3),
        "median": round(float(values.median()), 3),
        "std": round(float(values.std()), 3),
        "min": round(float(values.min()), 3),
        "max": round(float(values.max()), 3),
        "q25": round(float(values.quantile(0.25)), 3),
        "q75": round(float(values.quantile(0.75)), 3),
        "q95": round(float(values.quantile(0.95)), 3),
        "skew": round(float(values.skew()), 3),
        "kurtosis": round(float(values.kurtosis()), 3),
        "coefficient_of_variation": round(float(values.std() / values.mean()), 3),
        "units": cfg.TARGET_UNITS,
    }


def daily_profile(y: pd.Series) -> pd.DataFrame:
    """Mean, median and spread of the target by hour of day."""
    grouped = y.groupby(y.index.hour)
    return pd.DataFrame(
        {
            "hour": list(range(24)),
            "mean": grouped.mean().to_numpy(),
            "median": grouped.median().to_numpy(),
            "std": grouped.std().to_numpy(),
            "q25": grouped.quantile(0.25).to_numpy(),
            "q75": grouped.quantile(0.75).to_numpy(),
        }
    )


def weekly_profile(y: pd.Series) -> pd.DataFrame:
    """Mean and median of the target by day of week."""
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    grouped = y.groupby(y.index.dayofweek)
    return pd.DataFrame(
        {
            "dayofweek": list(range(7)),
            "day": labels,
            "mean": grouped.mean().to_numpy(),
            "median": grouped.median().to_numpy(),
            "std": grouped.std().to_numpy(),
        }
    )


def pattern_evidence(y: pd.Series) -> dict[str, Any]:
    """
    Quantify the daily and weekly patterns that the plots show visually.

    ``peak_to_trough_ratio`` compares the highest and lowest hourly means: a
    large ratio is direct evidence that time-of-day carries real signal, which
    in turn justifies both the seasonal period and the cyclical time features.
    """
    hourly = y.groupby(y.index.hour).mean()
    daily = y.groupby(y.index.dayofweek).mean()

    weekend = y[y.index.dayofweek >= 5]
    weekday = y[y.index.dayofweek < 5]

    return {
        "hourly_mean_min": round(float(hourly.min()), 2),
        "hourly_mean_min_hour": int(hourly.idxmin()),
        "hourly_mean_max": round(float(hourly.max()), 2),
        "hourly_mean_max_hour": int(hourly.idxmax()),
        "peak_to_trough_ratio_hourly": round(float(hourly.max() / hourly.min()), 3),
        "dayofweek_mean_min": round(float(daily.min()), 2),
        "dayofweek_mean_min_day": int(daily.idxmin()),
        "dayofweek_mean_max": round(float(daily.max()), 2),
        "dayofweek_mean_max_day": int(daily.idxmax()),
        "peak_to_trough_ratio_dayofweek": round(float(daily.max() / daily.min()), 3),
        "weekday_mean": round(float(weekday.mean()), 2),
        "weekend_mean": round(float(weekend.mean()), 2),
        "weekend_minus_weekday": round(float(weekend.mean() - weekday.mean()), 2),
        "note": (
            "A large hourly peak-to-trough ratio alongside a small day-of-week "
            "ratio indicates that the daily cycle dominates the weekly one."
        ),
    }


def correlation_with_target(
    data: pd.DataFrame, target: str = cfg.TARGET, top_n: int = 15
) -> pd.DataFrame:
    """
    Pearson correlation of every candidate regressor with the target.

    Computed on the training period only where the caller passes training data;
    used to motivate exogenous selection rather than including all 25 columns.
    """
    others = [c for c in data.columns if c != target]
    rows = [
        {
            "variable": col,
            "correlation": round(float(data[col].corr(data[target])), 4),
            "abs_correlation": round(abs(float(data[col].corr(data[target]))), 4),
            "group": (
                "weather" if col in cfg.WEATHER_COLS
                else "indoor" if col in cfg.INDOOR_COLS
                else "other_energy" if col in cfg.OTHER_ENERGY_COLS
                else "unclassified"
            ),
        }
        for col in others
    ]
    return (
        pd.DataFrame(rows)
        .sort_values("abs_correlation", ascending=False)
        .reset_index(drop=True)
        .head(top_n)
    )


def run_eda(data: pd.DataFrame, target: str = cfg.TARGET) -> dict[str, Any]:
    """Assemble every EDA output into one record for the analysis summary."""
    y = data[target]
    return {
        "summary_statistics": summary_statistics(y),
        "daily_profile": daily_profile(y).round(3).to_dict(orient="records"),
        "weekly_profile": weekly_profile(y).round(3).to_dict(orient="records"),
        "pattern_evidence": pattern_evidence(y),
        "top_correlations_with_target": correlation_with_target(data, target).to_dict(
            orient="records"
        ),
    }


---
## 4. Stationarity and seasonality (functions)

**Assignment Part 2.** ADF and KPSS test complementary hypotheses and are only conclusive read together. Differencing is applied only if the evidence supports it.

In [ ]:
"""
Stationarity, autocorrelation and seasonality analysis (assignment Part 2).

Guiding principle
-----------------
Differencing is applied only where the evidence supports it. The assignment
asks for stationarity to be *assessed*; it does not ask for the series to be
differenced regardless of what the tests say. Both ADF and KPSS are therefore
reported, because they test complementary hypotheses and are only conclusive
when read together:

    ADF  H0: a unit root is present      (reject  -> stationary)
    KPSS H0: the series is stationary    (reject  -> non-stationary)

    ADF rejects + KPSS does not reject -> stationary
    ADF fails   + KPSS rejects         -> non-stationary, difference
    both reject / neither rejects      -> ambiguous; inspect ACF and the
                                          seasonal structure before deciding
"""


from typing import Any, Sequence

import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf



# ------------------------------------------------------------------
# Unit-root tests
# ------------------------------------------------------------------


def adf_test(series: pd.Series, name: str = "series", regression: str = "c") -> dict[str, Any]:
    """
    Augmented Dickey-Fuller test.

    Reports the statistic, p-value, lag order chosen by AIC, the number of
    observations used, and the critical values, as required by Part 2 step 5.
    """
    values = pd.Series(series).dropna().to_numpy(dtype=float)
    stat, pvalue, used_lag, nobs, crit, _ = adfuller(
        values, autolag="AIC", regression=regression
    )
    return {
        "test": "ADF",
        "series": name,
        "null_hypothesis": "unit root present (non-stationary)",
        "statistic": float(stat),
        "p_value": float(pvalue),
        "used_lag": int(used_lag),
        "n_observations": int(nobs),
        "critical_values": {k: float(v) for k, v in crit.items()},
        "reject_null_5pct": bool(pvalue < 0.05),
        "conclusion": (
            "Reject the unit root at 5%: consistent with stationarity."
            if pvalue < 0.05
            else "Cannot reject the unit root at 5%: consistent with non-stationarity."
        ),
    }


def kpss_test(series: pd.Series, name: str = "series", regression: str = "c") -> dict[str, Any]:
    """
    KPSS test, whose null is the opposite of ADF's.

    ``statsmodels`` warns when the p-value is outside its interpolation table;
    the returned ``p_value_is_boundary`` flag records that so the number is not
    over-interpreted.
    """
    import warnings

    values = pd.Series(series).dropna().to_numpy(dtype=float)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        stat, pvalue, lags, crit = kpss(values, regression=regression, nlags="auto")
        boundary = any("p-value" in str(w.message) for w in caught)

    return {
        "test": "KPSS",
        "series": name,
        "null_hypothesis": "series is (trend/level) stationary",
        "statistic": float(stat),
        "p_value": float(pvalue),
        "p_value_is_boundary": bool(boundary),
        "lags": int(lags),
        "critical_values": {k: float(v) for k, v in crit.items()},
        "reject_null_5pct": bool(pvalue < 0.05),
        "conclusion": (
            "Reject stationarity at 5%: differencing may be needed."
            if pvalue < 0.05
            else "Cannot reject stationarity at 5%."
        ),
    }


def combined_verdict(adf: dict[str, Any], kpss_res: dict[str, Any]) -> dict[str, str]:
    """Cross-tabulate the two tests into a single, explicit recommendation."""
    adf_stationary = adf["reject_null_5pct"]
    kpss_stationary = not kpss_res["reject_null_5pct"]

    if adf_stationary and kpss_stationary:
        verdict, action = "stationary", "no differencing required"
    elif not adf_stationary and not kpss_stationary:
        verdict, action = "non-stationary", "apply first differencing"
    elif adf_stationary and not kpss_stationary:
        verdict, action = (
            "ambiguous (ADF says stationary, KPSS disagrees)",
            "possible trend-stationarity; inspect the decomposition before differencing",
        )
    else:
        verdict, action = (
            "ambiguous (KPSS says stationary, ADF disagrees)",
            "weak evidence of a unit root; prefer the model-selection result (AIC)",
        )

    return {"verdict": verdict, "recommended_action": action}


def assess_stationarity(
    series: pd.Series, name: str, difference_if_needed: bool = True
) -> dict[str, Any]:
    """
    Full stationarity assessment for one series, optionally repeated after
    differencing when (and only when) the evidence calls for it.
    """
    adf = adf_test(series, name)
    kp = kpss_test(series, name)
    verdict = combined_verdict(adf, kp)

    record: dict[str, Any] = {
        "series": name,
        "n": int(pd.Series(series).dropna().shape[0]),
        "adf": adf,
        "kpss": kp,
        **verdict,
        "differenced_tests": None,
    }

    needs_diff = verdict["verdict"] == "non-stationary"
    if difference_if_needed and needs_diff:
        diffed = pd.Series(series).diff().dropna()
        record["differenced_tests"] = {
            "transform": "first difference (d=1)",
            "adf": adf_test(diffed, f"diff({name})"),
            "kpss": kpss_test(diffed, f"diff({name})"),
        }
    record["differencing_applied_in_tests"] = bool(needs_diff)
    return record


# ------------------------------------------------------------------
# Autocorrelation
# ------------------------------------------------------------------


def acf_pacf_values(
    series: pd.Series, nlags: int = 168
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """ACF, PACF and the approximate 95% white-noise band."""
    values = pd.Series(series).dropna().to_numpy(dtype=float)
    acf_vals = acf(values, nlags=nlags, fft=True)
    pacf_vals = pacf(values, nlags=min(nlags, len(values) // 2 - 1), method="ywm")
    band = 1.96 / np.sqrt(len(values))
    return acf_vals, pacf_vals, band


def seasonality_evidence(
    series: pd.Series,
    daily: int = cfg.DAILY_PERIOD,
    weekly: int = cfg.WEEKLY_PERIOD,
    nlags: int = 200,
) -> dict[str, Any]:
    """
    Quantify daily and weekly structure directly from the autocorrelations.

    Compares the ACF at the seasonal lags and their multiples against the white
    noise band, and reports which lag in the searched range has the largest
    autocorrelation. This is the evidence behind the claim that s=24 is the
    appropriate seasonal period after hourly aggregation (Part 2, step 10).
    """
    acf_vals, _, band = acf_pacf_values(series, nlags=nlags)

    daily_multiples = [k * daily for k in range(1, nlags // daily + 1)]
    weekly_multiples = [k * weekly for k in range(1, max(nlags // weekly, 1) + 1)]

    non_seasonal = [
        lag for lag in range(1, min(nlags, len(acf_vals)))
        if lag % daily != 0
    ]
    peak_lag = int(np.argmax(acf_vals[1:]) + 1)

    return {
        "n_lags_examined": int(min(nlags, len(acf_vals) - 1)),
        "white_noise_band_95pct": float(band),
        "acf_at_daily_multiples": {
            int(lag): round(float(acf_vals[lag]), 4)
            for lag in daily_multiples if lag < len(acf_vals)
        },
        "acf_at_weekly_multiples": {
            int(lag): round(float(acf_vals[lag]), 4)
            for lag in weekly_multiples if lag < len(acf_vals)
        },
        "acf_lag1": round(float(acf_vals[1]), 4),
        "largest_acf_lag": peak_lag,
        "largest_acf_value": round(float(acf_vals[peak_lag]), 4),
        "mean_acf_at_daily_multiples": round(
            float(np.mean([acf_vals[l] for l in daily_multiples if l < len(acf_vals)])), 4
        ),
        "mean_acf_off_season": round(
            float(np.mean([acf_vals[l] for l in non_seasonal if l < len(acf_vals)])), 4
        ),
        "daily_peaks_exceed_band": bool(
            all(
                abs(acf_vals[lag]) > band
                for lag in daily_multiples[:3] if lag < len(acf_vals)
            )
        ),
    }


# ------------------------------------------------------------------
# Decomposition
# ------------------------------------------------------------------


def decompose(
    series: pd.Series, period: int = cfg.DAILY_PERIOD, method: str = "stl"
) -> Any:
    """
    Decompose the series into trend, seasonal and remainder components.

    STL is preferred over classical decomposition because it tolerates a
    seasonal pattern whose shape changes over the 4.5-month record, and its
    robust variant down-weights the large consumption spikes rather than
    letting them distort the seasonal component.
    """
    values = pd.Series(series).dropna()
    if method == "stl":
        return STL(values, period=period, robust=True).fit()
    return seasonal_decompose(values, model="additive", period=period)


def component_strengths(series: pd.Series, period: int = cfg.DAILY_PERIOD) -> dict[str, float]:
    """
    Wang-Smith-Hyndman trend and seasonal strength measures in [0, 1].

    ``Fs = max(0, 1 - Var(remainder) / Var(seasonal + remainder))``. Values near
    1 indicate strong structure; the conventional threshold for *seasonal
    differencing* is 0.64.
    """
    stl = decompose(series, period=period, method="stl")
    remainder_var = float(np.var(stl.resid))
    seasonal_plus = float(np.var(stl.seasonal + stl.resid))
    trend_plus = float(np.var(stl.trend + stl.resid))

    return {
        "period": int(period),
        "seasonal_strength_Fs": round(
            max(0.0, 1.0 - remainder_var / seasonal_plus) if seasonal_plus > 0 else 0.0, 4
        ),
        "trend_strength_Ft": round(
            max(0.0, 1.0 - remainder_var / trend_plus) if trend_plus > 0 else 0.0, 4
        ),
    }


def seasonal_period_justification(
    series: pd.Series, candidate_periods: Sequence[int] = (24, 168)
) -> dict[str, Any]:
    """
    Compare seasonal strength across candidate periods (Part 2, step 10).

    Explains *why* s=24 is the right seasonal period for the SARIMAX at hourly
    frequency: it is the strongest cycle, and s=168 would require 168 seasonal
    lags, which is computationally prohibitive and statistically fragile with
    only ~2,950 training observations (roughly 17 complete weeks).
    """
    strengths = {int(p): component_strengths(series, period=int(p)) for p in candidate_periods}
    best = max(strengths, key=lambda p: strengths[p]["seasonal_strength_Fs"])

    return {
        "strength_by_period": strengths,
        "strongest_period": int(best),
        "n_observations": int(pd.Series(series).dropna().shape[0]),
        "complete_weeks_available": round(pd.Series(series).dropna().shape[0] / 168.0, 1),
        "note": (
            "s=24 is used by SARIMAX. A weekly SARIMAX term (s=168) would add "
            "168 seasonal lags to the state vector, which is computationally "
            "prohibitive here; weekly structure is instead captured by the "
            "weekly seasonal naive benchmark and by lag_168 / roll_*_168 in the "
            "feature-based model."
        ),
    }


---
## 5. Benchmark models (functions)

**Assignment Part 4.** Each benchmark receives only the history available at the forecast origin, which makes leakage structurally impossible.

In [ ]:
"""
Benchmark forecasting models (assignment Part 4).

All five benchmarks share one interface,
``f(history: pd.Series, horizon: int) -> np.ndarray``, where ``history``
contains *only* observations available at the forecast origin. That signature
makes leakage structurally impossible: a benchmark cannot see the future
because it is never handed the future.

Formulae (Hyndman & Athanasopoulos, *FPP3*, ch. 5.2), with T = len(history):

* mean            : yhat_{T+h} = mean(y_1..y_T)
* naive           : yhat_{T+h} = y_T
* seasonal naive  : yhat_{T+h} = y_{T+h-m(k+1)},  k = floor((h-1)/m)
* drift           : yhat_{T+h} = y_T + h * (y_T - y_1) / (T - 1)
"""


from typing import Callable, Mapping

import numpy as np
import pandas as pd



# ------------------------------------------------------------------
# Point forecast functions
# ------------------------------------------------------------------


def mean_forecast(history: pd.Series, horizon: int) -> np.ndarray:
    """Flat forecast at the historical mean."""
    return np.repeat(float(np.mean(history)), horizon)


def naive_forecast(history: pd.Series, horizon: int) -> np.ndarray:
    """Flat forecast at the last observed value (random-walk optimal)."""
    return np.repeat(float(history.iloc[-1]), horizon)


def seasonal_naive_forecast(
    history: pd.Series, horizon: int, seasonality: int
) -> np.ndarray:
    """
    Seasonal naive: repeat the observation from the most recent same-phase slot.

    Closed form rather than recursive appending, so the mapping from step to
    source observation is explicit and testable. For h <= m this returns the
    value from exactly one season ago; for h > m it recycles the same season,
    which is the standard definition.
    """
    if seasonality <= 0:
        raise ValueError("seasonality must be positive")
    if len(history) < seasonality:
        raise ValueError(
            f"History of length {len(history)} is shorter than the seasonal "
            f"period {seasonality}."
        )

    values = np.asarray(history, dtype=float)
    out = np.empty(horizon, dtype=float)
    for h in range(1, horizon + 1):
        k = (h - 1) // seasonality
        out[h - 1] = values[len(values) + h - seasonality * (k + 1) - 1]
    return out


def drift_forecast(history: pd.Series, horizon: int) -> np.ndarray:
    """Random walk with drift equal to the average historical change."""
    if len(history) < 2:
        raise ValueError("Drift requires at least two observations.")
    y_first = float(history.iloc[0])
    y_last = float(history.iloc[-1])
    slope = (y_last - y_first) / (len(history) - 1)
    return y_last + slope * np.arange(1, horizon + 1, dtype=float)


def build_benchmark_registry(
    daily: int = cfg.DAILY_PERIOD, weekly: int = cfg.WEEKLY_PERIOD
) -> dict[str, Callable[[pd.Series, int], np.ndarray]]:
    """Return the five required benchmarks keyed by name."""
    return {
        "mean": mean_forecast,
        "naive": naive_forecast,
        "seasonal_naive_daily": lambda h_, n_: seasonal_naive_forecast(h_, n_, daily),
        "seasonal_naive_weekly": lambda h_, n_: seasonal_naive_forecast(h_, n_, weekly),
        "drift": drift_forecast,
    }


# ------------------------------------------------------------------
# Rolling-origin driver
# ------------------------------------------------------------------


def rolling_origin_benchmarks(
    y: pd.Series,
    test_index: pd.DatetimeIndex,
    horizon: int = cfg.HORIZON,
    registry: Mapping[str, Callable[[pd.Series, int], np.ndarray]] | None = None,
) -> tuple[dict[str, pd.Series], pd.DataFrame]:
    """
    Generate rolling-origin benchmark forecasts over the test period.

    At origin ``i`` the history is ``y`` up to (and excluding) the first
    timestamp of block ``i``: an expanding window that mimics an operator who
    re-forecasts every 24 hours with all data observed so far.

    Returns
    -------
    (forecasts, meta)
        ``forecasts`` maps model name -> forecast series over ``test_index``;
        ``meta`` records, for each test timestamp, its origin id and lead time.
    """
    registry = registry or build_benchmark_registry()
    n_origins = len(test_index) // horizon

    predictions: dict[str, list[float]] = {name: [] for name in registry}
    origins: list[int] = []
    steps: list[int] = []
    covered: list[pd.Timestamp] = []

    for origin in range(n_origins):
        block = test_index[origin * horizon : (origin + 1) * horizon]
        history = y.loc[y.index < block[0]]

        for name, fn in registry.items():
            predictions[name].extend(np.asarray(fn(history, horizon), dtype=float))

        covered.extend(list(block))
        origins.extend([origin] * horizon)
        steps.extend(range(1, horizon + 1))

    index = pd.DatetimeIndex(covered, name="date")
    forecasts = {
        name: pd.Series(values, index=index, name=name)
        for name, values in predictions.items()
    }
    meta = pd.DataFrame({"origin": origins, "step": steps}, index=index)
    return forecasts, meta


def strongest_benchmark(comparison: pd.DataFrame, metric: str = cfg.PRIMARY_METRIC) -> str:
    """
    Identify the best benchmark from an already-computed comparison table.

    Deliberately reads the *results* rather than hard-coding an expectation, so
    no claim about which benchmark wins is made before the code has run.
    """
    benchmark_names = set(build_benchmark_registry().keys())
    subset = comparison[comparison["model"].isin(benchmark_names)]
    if subset.empty:
        raise ValueError("No benchmark models found in the comparison table.")
    return str(subset.sort_values(metric).iloc[0]["model"])


---
## 6. Evaluation metrics (functions)

**Assignment Parts 3 and 9.** MAE, RMSE, MASE, Bias, sMAPE, plus the Diebold-Mariano test of equal predictive accuracy.

In [ ]:
"""
Forecast accuracy metrics and model comparison tests.

Covers assignment Parts 3 and 9.

Metric choices
--------------
* **MAE**   - average absolute error, in the units of the target.
* **RMSE**  - penalises large errors more heavily; relevant here because the
              cost of missing a large consumption spike is disproportionate.
* **MASE**  - MAE divided by the in-sample MAE of the *seasonal* naive
              forecast. Scale free, so it is comparable across series, and it
              has a natural reading: MASE < 1 beats the seasonal naive.
* **Bias**  - mean signed error (forecast minus actual); detects systematic
              over/under-forecasting, which MAE and RMSE hide.
* **sMAPE** - symmetric percentage error, reported as a secondary relative
              metric.

MAPE is deliberately **not** the headline metric. The target is strictly
positive here (min 10 Wh) so MAPE is computable, but it is heavily skewed by
the many low-consumption night-time hours and it penalises over-forecasting
more than under-forecasting.
"""


from typing import Any, Mapping

import numpy as np
import pandas as pd
from scipy import stats



# ------------------------------------------------------------------
# Individual metrics
# ------------------------------------------------------------------


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean absolute error."""
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Root mean squared error.

    Implemented directly rather than via ``mean_squared_error(squared=False)``,
    which was removed in scikit-learn 1.6.
    """
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def bias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean signed error: positive means the model over-forecasts."""
    return float(np.mean(y_pred - y_true))


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric mean absolute percentage error (%), guarded against 0/0."""
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    ratio = np.divide(
        np.abs(y_true - y_pred), denom,
        out=np.zeros_like(denom, dtype=float), where=denom != 0,
    )
    return float(100.0 * np.mean(ratio))


def mase_scale(y_train: pd.Series, seasonality: int = cfg.DAILY_PERIOD) -> float:
    """
    Denominator of MASE: in-sample MAE of the seasonal naive forecast.

    Computed on the **training** series only, so the scaling constant carries no
    test-period information.
    """
    values = np.asarray(y_train, dtype=float)
    if len(values) <= seasonality:
        raise ValueError("Training series shorter than the seasonal period.")
    return float(np.mean(np.abs(values[seasonality:] - values[:-seasonality])))


def mase(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    scale: float,
) -> float:
    """Mean absolute scaled error given a precomputed ``scale``."""
    if not np.isfinite(scale) or scale == 0:
        return float("nan")
    return float(np.mean(np.abs(y_true - y_pred)) / scale)


# ------------------------------------------------------------------
# Evaluation driver
# ------------------------------------------------------------------


def evaluate_forecast(
    name: str,
    y_true: pd.Series,
    y_pred: pd.Series,
    scale: float,
    extra: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    """
    Compute the full metric set for one model on aligned actual/forecast series.

    The two series are inner-joined on their index first, so a model that
    produces fewer points is never silently compared on a different sample; the
    number of points actually used is reported as ``n``.
    """
    joined = pd.concat(
        [y_true.rename("actual"), y_pred.rename("forecast")], axis=1, join="inner"
    ).dropna()

    if joined.empty:
        raise ValueError(f"No overlapping non-missing points for model '{name}'.")

    a = joined["actual"].to_numpy(dtype=float)
    f = joined["forecast"].to_numpy(dtype=float)

    record: dict[str, Any] = {
        "model": name,
        "n": int(len(joined)),
        "MAE": mae(a, f),
        "RMSE": rmse(a, f),
        "MASE": mase(a, f, scale),
        "Bias": bias(a, f),
        "sMAPE": smape(a, f),
    }
    if extra:
        record.update(dict(extra))
    return record


def build_comparison_table(
    forecasts: Mapping[str, pd.Series],
    y_true: pd.Series,
    y_train: pd.Series,
    seasonality: int = cfg.DAILY_PERIOD,
    sort_by: str = cfg.PRIMARY_METRIC,
    extras: Mapping[str, Mapping[str, Any]] | None = None,
) -> pd.DataFrame:
    """
    Evaluate every model on the identical test sample and rank by ``sort_by``.
    """
    scale = mase_scale(y_train, seasonality=seasonality)
    rows = []
    for name, pred in forecasts.items():
        rows.append(
            evaluate_forecast(
                name, y_true, pred, scale,
                extra=(extras or {}).get(name),
            )
        )
    table = pd.DataFrame(rows).sort_values(sort_by).reset_index(drop=True)
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    return table


def errors_by_horizon_step(
    forecasts: Mapping[str, pd.Series],
    y_true: pd.Series,
    step_index: pd.Series,
) -> pd.DataFrame:
    """
    RMSE as a function of the forecast lead time (1..H hours ahead).

    Reveals how quickly each model degrades with horizon, which a single pooled
    number hides. ``step_index`` maps each test timestamp to its lead time.
    """
    rows = []
    for name, pred in forecasts.items():
        joined = pd.concat(
            [y_true.rename("actual"), pred.rename("forecast"), step_index.rename("step")],
            axis=1, join="inner",
        ).dropna()
        for step, grp in joined.groupby("step"):
            rows.append(
                {
                    "model": name,
                    "step": int(step),
                    "RMSE": rmse(grp["actual"].to_numpy(float), grp["forecast"].to_numpy(float)),
                    "MAE": mae(grp["actual"].to_numpy(float), grp["forecast"].to_numpy(float)),
                }
            )
    return pd.DataFrame(rows)


# ------------------------------------------------------------------
# Statistical comparison
# ------------------------------------------------------------------


def diebold_mariano(
    y_true: pd.Series,
    pred_a: pd.Series,
    pred_b: pd.Series,
    h: int = cfg.HORIZON,
    loss: str = "squared",
) -> dict[str, Any]:
    """
    Diebold-Mariano test of equal predictive accuracy between two models.

    Answers "is model A's improvement over model B larger than sampling noise?"
    rather than relying on a raw metric difference. A Newey-West style variance
    with ``h - 1`` autocovariance lags accounts for the overlapping multi-step
    forecast errors, and the Harvey-Leybourne-Newbold small-sample correction is
    applied.

    Returns a record with the statistic, p-value and the sign convention:
    a **negative** statistic with a small p-value means A is more accurate.
    """
    joined = pd.concat(
        [y_true.rename("y"), pred_a.rename("a"), pred_b.rename("b")],
        axis=1, join="inner",
    ).dropna()

    e_a = joined["y"].to_numpy(float) - joined["a"].to_numpy(float)
    e_b = joined["y"].to_numpy(float) - joined["b"].to_numpy(float)

    if loss == "squared":
        d = e_a**2 - e_b**2
    elif loss == "absolute":
        d = np.abs(e_a) - np.abs(e_b)
    else:
        raise ValueError("loss must be 'squared' or 'absolute'")

    n = len(d)
    d_bar = float(np.mean(d))

    gamma0 = float(np.mean((d - d_bar) ** 2))
    var = gamma0
    for lag in range(1, h):
        if lag >= n:
            break
        cov = float(np.mean((d[lag:] - d_bar) * (d[:-lag] - d_bar)))
        var += 2.0 * cov
    var = max(var, 1e-12)

    dm = d_bar / np.sqrt(var / n)

    # Harvey-Leybourne-Newbold correction for overlapping horizons.
    correction = np.sqrt(max((n + 1 - 2 * h + h * (h - 1) / n) / n, 1e-12))
    dm_hln = dm * correction
    p_value = float(2 * (1 - stats.t.cdf(abs(dm_hln), df=n - 1)))

    return {
        "n": int(n),
        "loss": loss,
        "mean_loss_differential": d_bar,
        "dm_stat": float(dm_hln),
        "p_value": p_value,
        "significant_at_5pct": bool(p_value < 0.05),
        "more_accurate": "A" if d_bar < 0 else "B",
        "interpretation": (
            "Negative statistic favours model A. p < 0.05 means the accuracy "
            "difference is unlikely to be sampling noise."
        ),
    }


def pairwise_dm_against(
    y_true: pd.Series,
    forecasts: Mapping[str, pd.Series],
    reference: str,
    h: int = cfg.HORIZON,
) -> pd.DataFrame:
    """Diebold-Mariano test of every model against one reference model."""
    rows = []
    for name, pred in forecasts.items():
        if name == reference:
            continue
        res = diebold_mariano(y_true, pred, forecasts[reference], h=h)
        rows.append(
            {
                "model": name,
                "reference": reference,
                "dm_stat": round(res["dm_stat"], 3),
                "p_value": round(res["p_value"], 4),
                "significant_at_5pct": res["significant_at_5pct"],
                "beats_reference": bool(res["mean_loss_differential"] < 0),
            }
        )
    return pd.DataFrame(rows)


def skill_score(reference_metric: float, model_metric: float) -> float:
    """Percentage improvement of a model over a reference (positive = better)."""
    if reference_metric == 0:
        return float("nan")
    return float(100.0 * (reference_metric - model_metric) / reference_metric)


---
## 7. Feature engineering (functions)

**Assignment Part 6.** One builder serves both training and recursive inference, so the two paths cannot drift apart. Includes a programmatic leakage self-test.

In [ ]:
"""
Feature engineering for the machine-learning forecaster (assignment Part 6).

Leakage policy
--------------
Every feature is built by one function, :func:`make_feature_frame`, which is
used for **both** training-table construction and recursive multi-step
forecasting. Training and inference therefore cannot drift apart -- a common
and hard-to-spot source of leakage when the two paths are coded separately.

The rules enforced here:

1. Target lags use ``shift(lag)`` with ``lag >= 1``: row ``t`` never contains
   ``y_t``.
2. Rolling statistics are computed on ``y.shift(1)`` *before* the window is
   applied, so the window ending at row ``t`` spans ``y_{t-w} .. y_{t-1}`` and
   excludes ``y_t``.
3. Calendar features are deterministic functions of the timestamp. They are
   known arbitrarily far in advance and are always legitimate.
4. Sensor and weather regressors are handled under two explicitly named
   regimes:

   * ``conditional`` -- contemporaneous values at time ``t`` are used. This
     assumes perfect foresight of indoor sensors and outdoor weather, which no
     operator has. Forecasts under this regime are *conditional* forecasts and
     are labelled as such everywhere in the outputs.
   * ``operational`` -- only lags of at least ``HORIZON`` hours are used. With
     a 24-hour horizon and origin ``T``, the value at ``t - 24`` for any
     ``t <= T + 24`` satisfies ``t - 24 <= T`` and is therefore genuinely
     observed at the origin. Lag 1 would *not* be observed and is excluded.

Cyclical encoding
-----------------
Hour and day-of-week are circular: hour 23 is adjacent to hour 0, but a raw
integer encoding places them 23 units apart. Sine/cosine pairs restore that
adjacency. The raw integer versions are kept too because tree models can split
on them directly.
"""


from typing import Iterable, Sequence

import numpy as np
import pandas as pd


EXOG_REGIMES = ("conditional", "operational")


# ------------------------------------------------------------------
# Individual feature blocks
# ------------------------------------------------------------------


def add_time_features(frame: pd.DataFrame, index: pd.DatetimeIndex) -> pd.DataFrame:
    """Calendar and cyclical time-of-day / day-of-week features."""
    frame["hour"] = index.hour
    frame["dayofweek"] = index.dayofweek
    frame["month"] = index.month
    frame["is_weekend"] = (index.dayofweek >= 5).astype(int)

    frame["hour_sin"] = np.sin(2 * np.pi * index.hour / 24.0)
    frame["hour_cos"] = np.cos(2 * np.pi * index.hour / 24.0)
    frame["dow_sin"] = np.sin(2 * np.pi * index.dayofweek / 7.0)
    frame["dow_cos"] = np.cos(2 * np.pi * index.dayofweek / 7.0)
    return frame


def add_target_lags(
    frame: pd.DataFrame, y: pd.Series, lags: Sequence[int] = cfg.TARGET_LAGS
) -> pd.DataFrame:
    """Lagged target values. ``shift(lag)`` guarantees row t excludes y_t."""
    for lag in lags:
        frame[f"lag_{lag}"] = y.shift(lag)
    return frame


def add_rolling_features(
    frame: pd.DataFrame, y: pd.Series, windows: Sequence[int] = cfg.ROLLING_WINDOWS
) -> pd.DataFrame:
    """
    Rolling mean and standard deviation of the target.

    ``y.shift(1).rolling(w)`` is used, so the statistic at row ``t`` summarises
    ``y_{t-w} .. y_{t-1}``. Applying ``.rolling(w)`` first and shifting after
    would include ``y_t`` and leak the target.
    """
    shifted = y.shift(1)
    for window in windows:
        frame[f"roll_mean_{window}"] = shifted.rolling(window).mean()
        frame[f"roll_std_{window}"] = shifted.rolling(window).std()
    return frame


def add_exog_features(
    frame: pd.DataFrame,
    exog: pd.DataFrame,
    regime: str,
    exog_lags: Sequence[int] = cfg.EXOG_LAGS,
) -> pd.DataFrame:
    """Attach sensor/weather regressors under the requested availability regime."""
    if regime not in EXOG_REGIMES:
        raise ValueError(f"regime must be one of {EXOG_REGIMES}, got {regime!r}")

    if regime == "conditional":
        for col in exog.columns:
            frame[col] = exog[col]
    else:
        for col in exog.columns:
            for lag in exog_lags:
                frame[f"{col}_lag{lag}"] = exog[col].shift(lag)
    return frame


# ------------------------------------------------------------------
# Unified builder
# ------------------------------------------------------------------


def make_feature_frame(
    y: pd.Series,
    exog: pd.DataFrame | None = None,
    regime: str = "operational",
    target_lags: Sequence[int] = cfg.TARGET_LAGS,
    rolling_windows: Sequence[int] = cfg.ROLLING_WINDOWS,
    exog_lags: Sequence[int] = cfg.EXOG_LAGS,
    dropna: bool = True,
) -> pd.DataFrame:
    """
    Build the complete feature matrix for the timestamps in ``y.index``.

    This single function serves both training and recursive inference. During
    inference ``y`` is the *working* target series (observed history followed by
    the model's own predictions), so the same lag/rolling code path produces
    the future rows.

    Parameters
    ----------
    y : pd.Series
        Target series indexed by timestamp.
    exog : pd.DataFrame, optional
        Sensor/weather regressors aligned to ``y.index``.
    regime : {'conditional', 'operational'}
        Exogenous availability assumption; see module docstring.
    dropna : bool
        Drop warm-up rows that lack a full lag/rolling history. Set ``False``
        during recursive inference, where only the final row is needed.

    Returns
    -------
    pd.DataFrame
        Feature matrix (target column excluded).
    """
    index = pd.DatetimeIndex(y.index)
    frame = pd.DataFrame(index=index)

    frame = add_time_features(frame, index)
    frame = add_target_lags(frame, y, target_lags)
    frame = add_rolling_features(frame, y, rolling_windows)

    if exog is not None and not exog.empty:
        frame = add_exog_features(frame, exog.reindex(index), regime, exog_lags)

    return frame.dropna() if dropna else frame


def min_history_required(
    target_lags: Sequence[int] = cfg.TARGET_LAGS,
    rolling_windows: Sequence[int] = cfg.ROLLING_WINDOWS,
    exog_lags: Sequence[int] = cfg.EXOG_LAGS,
) -> int:
    """Number of past observations needed before a complete feature row exists."""
    return int(max(max(target_lags), max(rolling_windows) + 1, max(exog_lags)))


# ------------------------------------------------------------------
# Feature groups (for the Part 12 / Question 3 ablation)
# ------------------------------------------------------------------


def feature_groups(columns: Iterable[str]) -> dict[str, list[str]]:
    """
    Partition feature names into interpretable groups.

    Used both for the ablation study and for aggregating permutation
    importances into a story about *which kinds* of information matter.
    """
    columns = list(columns)
    time_cols = {
        "hour", "dayofweek", "month", "is_weekend",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    }

    groups: dict[str, list[str]] = {
        "time": [], "target_lags": [], "rolling": [], "weather": [], "indoor": [],
    }

    for col in columns:
        base = col.split("_lag")[0]
        if col in time_cols:
            groups["time"].append(col)
        elif col.startswith("lag_"):
            groups["target_lags"].append(col)
        elif col.startswith(("roll_mean_", "roll_std_")):
            groups["rolling"].append(col)
        elif base in cfg.WEATHER_COLS:
            groups["weather"].append(col)
        elif base in cfg.INDOOR_COLS or base in cfg.OTHER_ENERGY_COLS:
            groups["indoor"].append(col)

    return {k: v for k, v in groups.items() if v}


def ablation_specs() -> list[dict[str, object]]:
    """
    Cumulative feature sets for the Question 3 ablation.

    Each stage adds one group so the marginal contribution of that group is
    what changes between consecutive rows.
    """
    return [
        {"name": "time_only", "groups": ["time"]},
        {"name": "time+lags", "groups": ["time", "target_lags"]},
        {"name": "time+lags+rolling", "groups": ["time", "target_lags", "rolling"]},
        {"name": "time+lags+rolling+weather",
         "groups": ["time", "target_lags", "rolling", "weather"]},
        {"name": "all_features",
         "groups": ["time", "target_lags", "rolling", "weather", "indoor"]},
    ]


def columns_for_groups(all_columns: Iterable[str], groups: Sequence[str]) -> list[str]:
    """Resolve a list of group names to concrete column names."""
    mapping = feature_groups(all_columns)
    selected: list[str] = []
    for group in groups:
        selected.extend(mapping.get(group, []))
    return selected


# ------------------------------------------------------------------
# Leakage self-test
# ------------------------------------------------------------------


def leakage_self_test(y: pd.Series, exog: pd.DataFrame | None = None) -> dict[str, object]:
    """
    Programmatic check that no feature encodes the contemporaneous target.

    Perturbing ``y_t`` for a single timestamp must leave row ``t`` of the
    feature matrix unchanged. If any feature column moves, that column depends
    on the value being predicted and is leaking.

    Returns a record naming any offending columns (empty if clean).
    """
    regimes_checked: dict[str, object] = {}

    for regime in EXOG_REGIMES:
        base = make_feature_frame(y, exog, regime=regime, dropna=False)

        probe_pos = len(y) - cfg.HORIZON  # a row well inside the valid region
        probe_ts = y.index[probe_pos]

        perturbed = y.copy()
        perturbed.iloc[probe_pos] = float(perturbed.iloc[probe_pos]) + 1_000.0
        after = make_feature_frame(perturbed, exog, regime=regime, dropna=False)

        row_before = base.loc[probe_ts]
        row_after = after.loc[probe_ts]
        changed = [
            c for c in base.columns
            if not np.isclose(
                float(row_before[c]), float(row_after[c]), equal_nan=True, rtol=0, atol=1e-9
            )
        ]

        # Rows *after* the probe are legitimately allowed to change (that is
        # what a lag is for); only the probe row itself must be invariant.
        regimes_checked[regime] = {
            "probe_timestamp": str(probe_ts),
            "contemporaneous_target_leak": bool(changed),
            "leaking_columns": changed,
            "n_features": int(base.shape[1]),
        }

    return regimes_checked


---
## 8. Machine-learning model (functions)

**Assignment Part 7.** Gradient boosting with recursive multi-step forecasting and time-series-aware validation.

In [ ]:
"""
Feature-based machine-learning forecaster (assignment Part 7).

Model
-----
``HistGradientBoostingRegressor`` is used by default. XGBoost/LightGBM are
supported transparently if installed (see :func:`build_regressor`), but the
scikit-learn implementation is the reproducible default so the project runs
without optional dependencies.

Why recursive forecasting
-------------------------
The naive way to score a tree model on a hold-out period is to build the
feature table over the whole series and predict the test rows directly. That is
**not a forecast**: row ``T+24`` contains ``lag_1 = y_{T+23}``, an observation
that has not happened at the forecast origin. It silently converts a 24-hour
forecast into a sequence of 1-hour forecasts and flatters the model enormously.

:func:`recursive_forecast` instead rebuilds the feature row at every step from a
*working* target series consisting of observed history followed by the model's
own predictions. Target lags shorter than the horizon are therefore supplied by
the model itself, exactly as in deployment.

Exogenous regimes
-----------------
Both regimes are run and reported separately (see ``src/features.py``):

* ``operational`` -- sensor/weather values only at lags >= 24 h. Genuinely
  available at the origin. This is the honest headline result.
* ``conditional`` -- contemporaneous sensor/weather values. Assumes perfect
  foresight; reported as a conditional forecast and used to *quantify* the size
  of the advantage that perfect covariate knowledge confers.

Validation
----------
Hyper-parameters are chosen with :class:`sklearn.model_selection.TimeSeriesSplit`
on the **training period only**. There is no random cross-validation anywhere,
and the test period is never touched during selection.
"""


import time
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit



# ------------------------------------------------------------------
# Model construction
# ------------------------------------------------------------------


def available_backends() -> dict[str, bool]:
    """Report which gradient-boosting implementations are importable."""
    backends = {"hist_gbr": True}
    for name, module in (("xgboost", "xgboost"), ("lightgbm", "lightgbm")):
        try:
            __import__(module)
            backends[name] = True
        except ImportError:
            backends[name] = False
    return backends


def build_regressor(params: Mapping[str, Any] | None = None, backend: str = "hist_gbr"):
    """
    Construct the regressor for the requested backend.

    ``random_state`` is always set so that repeated runs are identical.
    """
    params = dict(params or {})
    if backend == "xgboost":
        import xgboost as xgb  # noqa: PLC0415

        return xgb.XGBRegressor(random_state=cfg.RANDOM_STATE, n_jobs=1, **params)
    if backend == "lightgbm":
        import lightgbm as lgb  # noqa: PLC0415

        return lgb.LGBMRegressor(random_state=cfg.RANDOM_STATE, n_jobs=1, verbose=-1, **params)
    return HistGradientBoostingRegressor(random_state=cfg.RANDOM_STATE, **params)


# ------------------------------------------------------------------
# Time-series-aware hyper-parameter selection
# ------------------------------------------------------------------


def select_hyperparameters(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    param_grid: Sequence[Mapping[str, Any]] = cfg.ML_PARAM_GRID,
    n_splits: int = cfg.ML_CV_SPLITS,
    backend: str = "hist_gbr",
) -> dict[str, Any]:
    """
    Choose hyper-parameters by expanding-window cross-validation.

    ``TimeSeriesSplit`` produces folds in which the validation block always
    follows the training block chronologically, preserving the arrow of time.
    Random K-fold would place future rows in the training fold and past rows in
    validation, which both leaks and mis-estimates generalisation.

    The grid is deliberately small (four candidates). Extensive tuning on a
    single short series risks selecting noise, and because selection never sees
    the test period, a larger grid would buy little.
    """
    # A gap of HORIZON-1 hours separates each training fold from its validation
    # block, so the fold boundary mirrors the real 24-hour-ahead task instead of
    # scoring the model on the hour immediately after the training data.
    splitter = TimeSeriesSplit(n_splits=n_splits, gap=cfg.HORIZON - 1)
    rows: list[dict[str, Any]] = []

    for i, params in enumerate(param_grid):
        fold_scores: list[float] = []
        for train_idx, valid_idx in splitter.split(X_train):
            model = build_regressor(params, backend=backend)
            model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
            pred = model.predict(X_train.iloc[valid_idx])
            fold_scores.append(rmse(y_train.iloc[valid_idx].to_numpy(float), pred))
        rows.append(
            {
                "candidate": i,
                "params": dict(params),
                "cv_rmse_mean": float(np.mean(fold_scores)),
                "cv_rmse_std": float(np.std(fold_scores)),
                "fold_rmse": [round(s, 3) for s in fold_scores],
            }
        )

    table = pd.DataFrame(rows).sort_values("cv_rmse_mean").reset_index(drop=True)
    best = table.iloc[0]

    return {
        "cv_table": table,
        "best_params": dict(best["params"]),
        "best_cv_rmse": float(best["cv_rmse_mean"]),
        "n_splits": n_splits,
        "gap_hours": cfg.HORIZON - 1,
        "note": (
            "Expanding-window TimeSeriesSplit on the training period only; "
            "one-step-ahead fit quality is used to rank candidates."
        ),
    }


# ------------------------------------------------------------------
# Training
# ------------------------------------------------------------------


def fit_ml_model(
    y_train: pd.Series,
    exog_train: pd.DataFrame | None,
    regime: str,
    params: Mapping[str, Any] | None = None,
    feature_subset: Sequence[str] | None = None,
    backend: str = "hist_gbr",
) -> dict[str, Any]:
    """
    Build the training table and fit the regressor.

    Returns the fitted model together with the exact feature column order, which
    :func:`recursive_forecast` must reproduce at inference time.
    """
    table = make_feature_frame(y_train, exog_train, regime=regime, dropna=True)
    if feature_subset is not None:
        table = table[list(feature_subset)]

    target = y_train.loc[table.index]

    t0 = time.time()
    model = build_regressor(params, backend=backend)
    model.fit(table, target)
    elapsed = time.time() - t0

    return {
        "model": model,
        "feature_columns": list(table.columns),
        "n_train_rows": int(len(table)),
        "regime": regime,
        "backend": backend,
        "params": dict(params or {}),
        "train_seconds": round(elapsed, 2),
        "train_start": str(table.index.min()),
        "train_end": str(table.index.max()),
    }


# ------------------------------------------------------------------
# Recursive multi-step forecasting
# ------------------------------------------------------------------


def recursive_forecast(
    fitted: Mapping[str, Any],
    y_history: pd.Series,
    test_index: pd.DatetimeIndex,
    exog_full: pd.DataFrame | None,
    horizon: int = cfg.HORIZON,
) -> pd.Series:
    """
    Rolling-origin, recursive ``horizon``-step-ahead forecasting.

    For each origin the working series is reset to the genuinely observed data
    (history plus every test observation *before* this origin), and predictions
    are then fed back in one step at a time. Resetting per origin is what makes
    this a sequence of 24-hour forecasts rather than one 336-step forecast:
    errors do not compound beyond a single day, which matches how a smart-home
    system would actually be operated.

    Parameters
    ----------
    fitted : mapping
        Output of :func:`fit_ml_model`.
    y_history : pd.Series
        Observed target over the training period.
    test_index : pd.DatetimeIndex
        Timestamps to forecast.
    exog_full : pd.DataFrame or None
        Exogenous frame covering both periods. Under the ``operational`` regime
        only lagged values are read from it; under ``conditional`` the
        contemporaneous test-period rows are read and the forecast is
        conditional by construction.
    """
    model = fitted["model"]
    columns = fitted["feature_columns"]
    regime = fitted["regime"]

    n_origins = len(test_index) // horizon
    warm = min_history_required() + max(cfg.ROLLING_WINDOWS) + 2

    predictions: list[float] = []
    covered: list[pd.Timestamp] = []

    # Full observed target (train + test actuals). Test actuals are used ONLY to
    # reset the working series at each new origin, i.e. to represent data that
    # has genuinely been observed by the time that origin arrives. They are
    # never used inside a block.
    observed = y_history.copy()

    for origin in range(n_origins):
        block = test_index[origin * horizon : (origin + 1) * horizon]
        working = observed.loc[observed.index < block[0]].copy()

        for step, timestamp in enumerate(block, start=1):
            working.loc[timestamp] = np.nan  # placeholder row to be predicted
            tail = working.iloc[-warm:]

            exog_tail = (
                exog_full.reindex(tail.index) if exog_full is not None else None
            )
            row = make_feature_frame(
                tail, exog_tail, regime=regime, dropna=False
            ).loc[[timestamp], columns]

            if row.isna().any(axis=None):
                missing = row.columns[row.isna().iloc[0]].tolist()
                raise ValueError(
                    f"Incomplete feature row at {timestamp} (step {step}): {missing}"
                )

            yhat = float(model.predict(row)[0])
            working.loc[timestamp] = yhat  # feed the prediction back in
            predictions.append(yhat)
            covered.append(timestamp)

    return pd.Series(predictions, index=pd.DatetimeIndex(covered, name="date"))


def attach_observed_test(y_history: pd.Series, y_test: pd.Series) -> pd.Series:
    """Concatenate training history and observed test values for origin resets."""
    return pd.concat([y_history, y_test]).sort_index()


# ------------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------------


def compute_permutation_importance(
    fitted: Mapping[str, Any],
    y_train: pd.Series,
    exog_train: pd.DataFrame | None,
    holdout_hours: int = cfg.TEST_STEPS,
    n_repeats: int = 5,
) -> pd.DataFrame:
    """
    Permutation importance on a held-out *tail of the training period*.

    Importance is deliberately **not** computed on the test period: doing so
    would use test data to interpret (and potentially to select) the model. The
    final ``holdout_hours`` of training data act as a clean proxy.

    Permutation importance is preferred over a tree's internal split counts,
    which are biased towards high-cardinality continuous features.
    """
    model = fitted["model"]
    columns = fitted["feature_columns"]

    table = make_feature_frame(
        y_train, exog_train, regime=fitted["regime"], dropna=True
    )[columns]
    target = y_train.loc[table.index]

    X_holdout = table.iloc[-holdout_hours:]
    y_holdout = target.iloc[-holdout_hours:]

    result = permutation_importance(
        model, X_holdout, y_holdout,
        n_repeats=n_repeats,
        random_state=cfg.RANDOM_STATE,
        scoring="neg_root_mean_squared_error",
    )

    return (
        pd.DataFrame(
            {
                "feature": columns,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std,
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


def aggregate_importance_by_group(importance: pd.DataFrame) -> pd.DataFrame:
    """Sum permutation importances within each feature group."""
    mapping = feature_groups(importance["feature"])
    lookup = {col: group for group, cols in mapping.items() for col in cols}
    out = importance.copy()
    out["group"] = out["feature"].map(lookup).fillna("other")
    return (
        out.groupby("group", as_index=False)["importance_mean"]
        .sum()
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


def run_feature_ablation(
    y_train: pd.Series,
    y_test: pd.Series,
    exog_train: pd.DataFrame | None,
    exog_full: pd.DataFrame | None,
    regime: str,
    params: Mapping[str, Any],
    mase_scale_value: float,
    horizon: int = cfg.HORIZON,
) -> pd.DataFrame:
    """
    Cumulative feature-group ablation (assignment Question 3).

    Each stage adds one group of features and is evaluated with the *same*
    recursive rolling-origin protocol as the headline model, so the differences
    between rows are attributable to the features rather than to the evaluation
    procedure.
    """

    full_table = make_feature_frame(y_train, exog_train, regime=regime, dropna=True)
    observed = attach_observed_test(y_train, y_test)

    rows: list[dict[str, Any]] = []
    for spec in ablation_specs():
        subset = columns_for_groups(full_table.columns, spec["groups"])
        if not subset:
            continue

        fitted = fit_ml_model(
            y_train, exog_train, regime=regime, params=params, feature_subset=subset
        )
        pred = recursive_forecast(
            fitted, observed, y_test.index, exog_full, horizon=horizon
        )
        aligned = pd.concat(
            [y_test.rename("a"), pred.rename("f")], axis=1, join="inner"
        ).dropna()

        rows.append(
            {
                "feature_set": spec["name"],
                "n_features": len(subset),
                "MAE": mae(aligned["a"].to_numpy(float), aligned["f"].to_numpy(float)),
                "RMSE": rmse(aligned["a"].to_numpy(float), aligned["f"].to_numpy(float)),
                "MASE": mase(
                    aligned["a"].to_numpy(float), aligned["f"].to_numpy(float),
                    mase_scale_value,
                ),
            }
        )

    table = pd.DataFrame(rows)
    table["RMSE_change_vs_previous"] = table["RMSE"].diff()
    return table


---
## 9. SARIMA / SARIMAX (functions)

**Assignment Part 5.** AIC grid search over the full required space, evidence-based exogenous selection, residual diagnostics.

In [ ]:
"""
SARIMA / SARIMAX: order selection by AIC, exogenous variable selection,
residual diagnostics and rolling-origin forecasting.

Covers assignment Part 5.

Methodological notes
--------------------
1. **AIC comparability across differencing orders.** ``statsmodels``' state
   space SARIMAX uses ``simple_differencing=False`` by default, so the number
   of observations entering the likelihood is the same for every (d, D). AIC
   values are therefore directly comparable across differencing orders. This is
   asserted programmatically by :func:`verify_aic_comparability` rather than
   assumed.

2. **Staged search.** The complete required non-seasonal grid
   (p in [0,6], d in [0,2], q in [0,6]) is searched; the seasonal grid is then
   crossed against the best non-seasonal candidates. See ``config.py`` for the
   timing measurements that motivate this and note that no part of the required
   (p, d, q) space is dropped.

3. **Parameters are estimated on training data only.** During rolling-origin
   evaluation the fitted model is *filtered* forward through newly observed
   test data with ``append(..., refit=False)``. This updates the state (which a
   real operator would also have) but never re-estimates coefficients on test
   data.
"""


import gc
import itertools
import signal
import time
import warnings
from contextlib import contextmanager
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX


warnings.filterwarnings("ignore")


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------


def _fit_one(
    y: pd.Series,
    order: tuple[int, int, int],
    seasonal_order: tuple[int, int, int, int],
    exog: pd.DataFrame | None = None,
    maxiter: int = cfg.GRID_MAXITER,
    trend: str = cfg.SARIMAX_TREND,
) -> Any:
    """Fit a single SARIMAX. Raises on failure; callers decide how to handle."""
    # A constant trend is not identified when the series is differenced, so it
    # is dropped automatically in that case (a common source of silent failure).
    effective_trend = trend if (order[1] == 0 and seasonal_order[1] == 0) else "n"

    model = SARIMAX(
        y,
        exog=exog,
        order=order,
        seasonal_order=seasonal_order,
        trend=effective_trend,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False, maxiter=maxiter)


class _FitTimeout(Exception):
    """Raised when a single SARIMAX fit exceeds its wall-clock budget."""


@contextmanager
def _fit_time_limit(seconds: int):
    """
    Abort a single model fit that exceeds ``seconds``.

    Some high-order specifications at s=24 (e.g. (6,1,6)(0,1,1,24), whose state
    vector spans 6 + 6 + 24 + 24 lags) can run for many minutes without
    converging. Without a cap, one such model stalls the whole grid. Models that
    hit the cap are recorded with status ``timeout`` rather than silently
    dropped, so the search remains auditable.

    Uses SIGALRM, which only interrupts the main thread on POSIX systems; on
    other platforms the limit is a no-op and fits simply run to completion.
    """
    if seconds is None or seconds <= 0 or not hasattr(signal, "SIGALRM"):
        yield
        return

    def _handler(signum, frame):  # noqa: ARG001
        raise _FitTimeout(f"fit exceeded {seconds}s")

    previous = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(int(seconds))
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, previous)


def fit_sarimax_safe(
    y: pd.Series,
    order: tuple[int, int, int],
    seasonal_order: tuple[int, int, int, int] = (0, 0, 0, 0),
    exog: pd.DataFrame | None = None,
    maxiter: int = cfg.GRID_MAXITER,
    timeout: int = cfg.GRID_FIT_TIMEOUT_S,
) -> dict[str, Any]:
    """
    Fit one SARIMAX and return a result record, never raising.

    A single failed, non-converging or pathologically slow specification must
    not terminate the search (assignment Part 5), so every exception -- and the
    wall-clock timeout -- is caught and recorded.
    """
    record: dict[str, Any] = {
        "p": order[0], "d": order[1], "q": order[2],
        "P": seasonal_order[0], "D": seasonal_order[1],
        "Q": seasonal_order[2], "s": seasonal_order[3],
        "order": str(order), "seasonal_order": str(seasonal_order),
        "aic": np.nan, "bic": np.nan, "hqic": np.nan, "llf": np.nan,
        "nobs": np.nan, "n_params": np.nan,
        "converged": False, "status": "failed", "error": "",
        "fit_seconds": np.nan,
    }

    start = time.time()
    try:
        with warnings.catch_warnings(), _fit_time_limit(timeout):
            warnings.simplefilter("ignore")
            res = _fit_one(y, order, seasonal_order, exog=exog, maxiter=maxiter)

        aic = float(res.aic)
        if not np.isfinite(aic):
            record["status"] = "non-finite AIC"
            record["error"] = "AIC is NaN/inf (degenerate likelihood)"
        else:
            record["aic"] = aic
            record["bic"] = float(res.bic)
            record["hqic"] = float(res.hqic)
            record["llf"] = float(res.llf)
            record["nobs"] = int(res.nobs)
            record["n_params"] = int(len(res.params))
            record["converged"] = bool(res.mle_retvals.get("converged", False))
            record["status"] = "ok" if record["converged"] else "not converged"
    except _FitTimeout as exc:
        record["status"] = "timeout"
        record["error"] = str(exc)
    except Exception as exc:  # noqa: BLE001 - deliberate: search must continue
        record["status"] = "failed"
        record["error"] = f"{type(exc).__name__}: {exc}"[:200]

    record["fit_seconds"] = round(time.time() - start, 2)
    return record


GRID_CACHE_PATH = cfg.METRICS_DIR / "sarimax_grid_search.csv"


def _cache_key(order: tuple[int, int, int], seasonal_order: tuple[int, int, int, int]) -> str:
    return f"{order}|{seasonal_order}"


def load_grid_cache(path: Any = None) -> pd.DataFrame:
    """Load previously computed grid records, if any."""
    path = path or GRID_CACHE_PATH
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def _append_cache(record: dict[str, Any], path: Any = None) -> None:
    """
    Append one fitted-model record to the on-disk cache immediately.

    Writing incrementally makes the search resumable: a long grid can be
    interrupted (or run in chunks on a constrained machine) and picked up
    exactly where it stopped, without refitting anything.
    """
    path = path or GRID_CACHE_PATH
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame([record])
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def _cached_keys(path: Any = None) -> set[str]:
    cache = load_grid_cache(path)
    if cache.empty:
        return set()
    return {
        _cache_key(eval(o), eval(s))  # noqa: S307 - values written by this module
        for o, s in zip(cache["order"], cache["seasonal_order"])
    }


def verify_aic_comparability(y: pd.Series) -> dict[str, Any]:
    """
    Empirically confirm that ``nobs`` (and hence AIC) is invariant to d and D.

    Returns the observed ``nobs`` for a few differencing settings.
    """
    checks = {}
    for d, D in [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1)]:
        rec = fit_sarimax_safe(y, (1, d, 1), (0, D, 0, cfg.SEASONAL_PERIOD), maxiter=15)
        checks[f"d={d},D={D}"] = rec["nobs"]
    values = [v for v in checks.values() if v == v]  # drop NaN
    return {
        "nobs_by_differencing": checks,
        "aic_comparable_across_d": bool(len(set(values)) == 1),
    }


def decide_seasonal_differencing(
    y: pd.Series, period: int = cfg.SEASONAL_PERIOD, threshold: float = 0.64
) -> dict[str, Any]:
    """
    Choose D from seasonal-strength evidence rather than by assumption.

    Uses the Wang-Smith-Hyndman seasonal strength measure,
    ``Fs = max(0, 1 - Var(remainder) / Var(seasonal + remainder))`` computed on
    an STL decomposition. The conventional rule of thumb is that seasonal
    differencing is warranted when ``Fs > 0.64``.

    Stage 2 of the grid search still tests D in {0, 1} empirically, so this is a
    starting point, not a final decision.
    """
    stl = STL(y, period=period, robust=True).fit()
    remainder_var = float(np.var(stl.resid))
    seasonal_plus = float(np.var(stl.seasonal + stl.resid))
    strength = max(0.0, 1.0 - remainder_var / seasonal_plus) if seasonal_plus > 0 else 0.0

    trend_plus = float(np.var(stl.trend + stl.resid))
    trend_strength = max(0.0, 1.0 - remainder_var / trend_plus) if trend_plus > 0 else 0.0

    return {
        "seasonal_strength_Fs": round(strength, 4),
        "trend_strength_Ft": round(trend_strength, 4),
        "threshold": threshold,
        "recommended_D": int(strength > threshold),
        "rule": "Wang-Smith-Hyndman: seasonally difference when Fs > 0.64",
    }


# ------------------------------------------------------------------
# Grid search
# ------------------------------------------------------------------


def grid_search_stage1(
    y: pd.Series,
    p_values: Sequence[int] = cfg.SEARCH_P,
    d_values: Sequence[int] = cfg.SEARCH_D,
    q_values: Sequence[int] = cfg.SEARCH_Q,
    seasonal_order: tuple[int, int, int, int] | None = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Search the complete required non-seasonal grid, ranked by AIC.

    The seasonal structure is held fixed at ``(0, D*, 0, s)`` so that the
    comparison isolates the effect of (p, d, q).
    """
    if seasonal_order is None:
        seasonal_order = (0, 1, 0, cfg.SEASONAL_PERIOD)

    combos = list(itertools.product(p_values, d_values, q_values))
    done = _cached_keys()

    if verbose:
        print(
            f"[sarimax] Stage 1: {len(combos)} non-seasonal orders "
            f"with seasonal_order={seasonal_order} "
            f"({len(done)} already cached)"
        )

    t0 = time.time()
    for i, (p, d, q) in enumerate(combos, start=1):
        key = _cache_key((p, d, q), seasonal_order)
        if key in done:
            continue
        rec = fit_sarimax_safe(y, (p, d, q), seasonal_order)
        rec["stage"] = 1
        _append_cache(rec)
        if verbose:
            print(
                f"[sarimax]  ({i:3d}/{len(combos)}) "
                f"({p},{d},{q}){seasonal_order} "
                f"aic={rec['aic']:.1f} status={rec['status']} "
                f"t={rec['fit_seconds']}s",
                flush=True,
            )

    if verbose:
        print(f"[sarimax] Stage 1 pass finished in {time.time() - t0:.0f}s")

    cache = load_grid_cache()
    stage1 = cache[cache["stage"] == 1]
    return stage1.sort_values("aic", na_position="last").reset_index(drop=True)


def grid_search_stage2(
    y: pd.Series,
    top_orders: Iterable[tuple[int, int, int]],
    seasonal_P: Sequence[int] = cfg.SEARCH_SEASONAL_P,
    seasonal_D: Sequence[int] = cfg.SEARCH_SEASONAL_D,
    seasonal_Q: Sequence[int] = cfg.SEARCH_SEASONAL_Q,
    s: int = cfg.SEASONAL_PERIOD,
    verbose: bool = True,
) -> pd.DataFrame:
    """Cross the promoted non-seasonal orders with the seasonal grid."""
    top_orders = list(top_orders)
    seasonal_combos = [
        c for c in itertools.product(seasonal_P, seasonal_D, seasonal_Q) if c != (0, 0, 0)
    ]
    total = len(top_orders) * len(seasonal_combos)
    done = _cached_keys()

    if verbose:
        print(
            f"[sarimax] Stage 2: {total} seasonal specifications at s={s} "
            f"({len(done)} records already cached)"
        )

    t0 = time.time()
    i = 0
    for order in top_orders:
        for P, D, Q in seasonal_combos:
            i += 1
            if _cache_key(order, (P, D, Q, s)) in done:
                continue
            rec = fit_sarimax_safe(y, order, (P, D, Q, s))
            rec["stage"] = 2
            _append_cache(rec)
            if verbose:
                print(
                    f"[sarimax]  ({i:3d}/{total}) {order}({P},{D},{Q},{s}) "
                    f"aic={rec['aic']:.1f} status={rec['status']} "
                    f"t={rec['fit_seconds']}s",
                    flush=True,
                )

    if verbose:
        print(f"[sarimax] Stage 2 pass finished in {time.time() - t0:.0f}s")

    cache = load_grid_cache()
    stage2 = cache[cache["stage"] == 2]
    return stage2.sort_values("aic", na_position="last").reset_index(drop=True)


def run_full_grid_search(y: pd.Series, save: bool = True) -> dict[str, Any]:
    """
    Execute the staged AIC grid search and return the selected specification.
    """
    cfg.ensure_dirs()
    t0 = time.time()

    comparability = verify_aic_comparability(y)
    print(f"[sarimax] AIC comparability check: {comparability}")

    d_decision = decide_seasonal_differencing(y)
    D_star = d_decision["recommended_D"]
    print(f"[sarimax] Seasonal differencing evidence: {d_decision}")

    stage1 = grid_search_stage1(
        y, seasonal_order=(0, D_star, 0, cfg.SEASONAL_PERIOD)
    )
    valid1 = stage1[stage1["status"] != "failed"].dropna(subset=["aic"])
    top_orders = [
        (int(r.p), int(r.d), int(r.q))
        for r in valid1.head(cfg.GRID_TOP_K).itertuples()
    ]
    print(f"[sarimax] Promoting to stage 2: {top_orders}")

    grid_search_stage2(y, top_orders)

    # The incremental cache is the single source of truth for both stages.
    results = load_grid_cache().sort_values("aic", na_position="last").reset_index(drop=True)

    valid = results[(results["status"] == "ok")].dropna(subset=["aic"])
    if valid.empty:
        valid = results.dropna(subset=["aic"])
        print("[sarimax] WARNING: no fully converged model; using best finite-AIC model.")

    best = valid.iloc[0]
    best_order = (int(best.p), int(best.d), int(best.q))
    best_seasonal = (int(best.P), int(best.D), int(best.Q), int(best.s))

    if save:
        path = cfg.METRICS_DIR / "sarimax_grid_search_ranked.csv"
        results.to_csv(path, index=False)
        print(f"[sarimax] Ranked grid results -> {path}")

    elapsed = time.time() - t0
    print(
        f"[sarimax] Selected {best_order}{best_seasonal} "
        f"AIC={best.aic:.2f} (search took {elapsed:.0f}s)"
    )

    return {
        "results": results,
        "best_order": best_order,
        "best_seasonal_order": best_seasonal,
        "best_aic": float(best.aic),
        "best_bic": float(best.bic),
        "aic_comparability": comparability,
        "seasonal_differencing_decision": d_decision,
        "n_models_attempted": int(len(results)),
        "n_models_converged": int((results["status"] == "ok").sum()),
        "n_models_failed": int((results["status"] == "failed").sum()),
        "search_seconds": round(elapsed, 1),
        "grid_definition": {
            "p": list(cfg.SEARCH_P), "d": list(cfg.SEARCH_D), "q": list(cfg.SEARCH_Q),
            "P": list(cfg.SEARCH_SEASONAL_P), "D": list(cfg.SEARCH_SEASONAL_D),
            "Q": list(cfg.SEARCH_SEASONAL_Q), "s": cfg.SEASONAL_PERIOD,
        },
    }


# ------------------------------------------------------------------
# Exogenous variable selection
# ------------------------------------------------------------------


def select_exog(
    data: pd.DataFrame,
    y_train: pd.Series,
    candidates: Sequence[str] = cfg.SARIMAX_CANDIDATE_EXOG,
    corr_threshold: float = 0.05,
    vif_threshold: float = 10.0,
) -> dict[str, Any]:
    """
    Select SARIMAX exogenous regressors on evidence, not by including everything.

    Two screens are applied, both computed on the **training period only**:

    1. *Relevance*: absolute Pearson correlation with the target must exceed
       ``corr_threshold``. Regressors unrelated to the target only add
       parameters and inflate AIC.
    2. *Redundancy*: variance inflation factors are computed and the most
       collinear variable is dropped iteratively while any VIF exceeds
       ``vif_threshold``. ``T_out`` and ``Tdewpoint`` in particular are close to
       collinear, which makes SARIMAX coefficients unstable and uninterpretable.

    Returns a record documenting every decision.
    """
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    available = [c for c in candidates if c in data.columns]
    X = data.loc[y_train.index, available].astype(float)

    corrs = {c: float(X[c].corr(y_train)) for c in available}
    relevant = [c for c in available if abs(corrs[c]) >= corr_threshold]
    dropped_irrelevant = [c for c in available if c not in relevant]

    kept = list(relevant)
    dropped_collinear: list[dict[str, Any]] = []
    vif_history: list[dict[str, float]] = []

    while len(kept) > 1:
        Xk = X[kept].copy()
        Xk.insert(0, "const", 1.0)
        vifs = {
            kept[i - 1]: float(variance_inflation_factor(Xk.values, i))
            for i in range(1, Xk.shape[1])
        }
        vif_history.append({k: round(v, 2) for k, v in vifs.items()})
        worst = max(vifs, key=vifs.get)
        if vifs[worst] <= vif_threshold:
            break
        dropped_collinear.append({"variable": worst, "vif": round(vifs[worst], 2)})
        kept.remove(worst)

    return {
        "candidates": available,
        "correlations_with_target": {k: round(v, 4) for k, v in corrs.items()},
        "dropped_low_correlation": dropped_irrelevant,
        "corr_threshold": corr_threshold,
        "vif_threshold": vif_threshold,
        "vif_history": vif_history,
        "dropped_collinear": dropped_collinear,
        "selected": kept,
        "note": (
            "Selection statistics are computed on the training period only. "
            "Using the exogenous values of the test period at forecast time "
            "makes the resulting forecast CONDITIONAL, not operational."
        ),
    }


# ------------------------------------------------------------------
# Final fit, diagnostics, forecasting
# ------------------------------------------------------------------


def fit_final_sarimax(
    y_train: pd.Series,
    order: tuple[int, int, int],
    seasonal_order: tuple[int, int, int, int],
    exog_train: pd.DataFrame | None = None,
    maxiter: int = 200,
    cache_key: str | None = None,
) -> Any:
    """
    Refit the selected specification to convergence on the training data.

    When ``cache_key`` is given the fitted result is pickled to
    ``outputs/models/`` and reloaded on subsequent runs. Refitting a
    (5,0,3)(0,1,1,24) model takes around 100 seconds, so caching makes repeated
    pipeline runs practical without changing any result.
    """
    cache_path = cfg.MODEL_DIR / f"{cache_key}.pkl" if cache_key else None

    if cache_path is not None and cache_path.exists():
        from statsmodels.tsa.statespace.sarimax import SARIMAXResults

        try:
            res = SARIMAXResults.load(cache_path)
            print(f"[sarimax] Loaded cached fit {order}{seasonal_order} from {cache_path.name}")
            return res
        except Exception as exc:  # noqa: BLE001
            print(f"[sarimax] Cache unreadable ({exc}); refitting.")

    t0 = time.time()
    res = _fit_one(y_train, order, seasonal_order, exog=exog_train, maxiter=maxiter)
    print(
        f"[sarimax] Final fit {order}{seasonal_order} "
        f"exog={'yes' if exog_train is not None else 'no'} "
        f"AIC={res.aic:.2f} in {time.time() - t0:.0f}s"
    )

    if cache_path is not None:
        cfg.ensure_dirs()
        try:
            res.save(cache_path)
        except Exception as exc:  # noqa: BLE001
            print(f"[sarimax] Could not cache fit: {exc}")

    return res


def residual_diagnostics(res: Any, lags: int = 48) -> dict[str, Any]:
    """
    Residual diagnostics for the fitted SARIMAX (assignment Part 5).

    What each test checks
    ---------------------
    * **Ljung-Box** -- whether residual autocorrelation remains at the tested
      lags. A small p-value means structure is left in the residuals, i.e. the
      model has not captured all the serial dependence.
    * **Jarque-Bera** -- whether residuals are Gaussian. Non-normal residuals do
      not bias the point forecasts but do make the analytic prediction
      intervals unreliable.
    * **Heteroskedasticity** -- whether residual variance is stable over time.
      A failure means the intervals are too narrow in volatile periods.
    * **Summary statistics** -- mean near zero indicates an unbiased in-sample
      fit; skew/kurtosis quantify the departure from normality.
    """
    resid = pd.Series(res.resid).dropna()
    # Drop the diffuse-initialisation burn-in, which is not a genuine residual.
    burn = max(cfg.SEASONAL_PERIOD, 24)
    resid = resid.iloc[burn:]

    lb = acorr_ljungbox(resid, lags=[12, 24, 48], return_df=True)
    lb_rows = [
        {"lag": int(lag), "lb_stat": float(r.lb_stat), "lb_pvalue": float(r.lb_pvalue)}
        for lag, r in lb.iterrows()
    ]

    try:
        jb_stat, jb_p, skew, kurt = res.test_normality(method="jarquebera")[0]
    except Exception:  # noqa: BLE001
        jb_stat = jb_p = skew = kurt = float("nan")

    try:
        het_stat, het_p = res.test_heteroskedasticity(method="breakvar")[0]
    except Exception:  # noqa: BLE001
        het_stat = het_p = float("nan")

    return {
        "n_residuals_used": int(len(resid)),
        "burn_in_dropped": burn,
        "mean": float(resid.mean()),
        "std": float(resid.std()),
        "min": float(resid.min()),
        "max": float(resid.max()),
        "skew": float(resid.skew()),
        "kurtosis": float(resid.kurtosis()),
        "ljung_box": lb_rows,
        "ljung_box_lag24_pvalue": float(lb.loc[24, "lb_pvalue"]),
        "residual_autocorrelation_remains": bool(lb.loc[24, "lb_pvalue"] < 0.05),
        "jarque_bera_stat": float(jb_stat),
        "jarque_bera_pvalue": float(jb_p),
        "residuals_normal": bool(jb_p > 0.05) if jb_p == jb_p else None,
        "heteroskedasticity_stat": float(het_stat),
        "heteroskedasticity_pvalue": float(het_p),
        "homoskedastic": bool(het_p > 0.05) if het_p == het_p else None,
    }


def rolling_origin_forecast(
    res: Any,
    y_test: pd.Series,
    horizon: int = cfg.HORIZON,
    exog_test: pd.DataFrame | None = None,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Produce rolling-origin ``horizon``-step forecasts across the test period.

    At each origin the model state is filtered forward through the observations
    that have *already happened*, using ``append(..., refit=False)``. Model
    coefficients stay fixed at their training-data estimates, so no test
    information enters parameter estimation.

    Returns
    -------
    DataFrame indexed by timestamp with columns
    ``[forecast, lower, upper, origin, step]``.
    """
    n_origins = len(y_test) // horizon
    current = res
    blocks: list[pd.DataFrame] = []

    for origin in range(n_origins):
        start = origin * horizon
        stop = start + horizon
        idx = y_test.index[start:stop]

        exog_block = exog_test.iloc[start:stop] if exog_test is not None else None
        fc = current.get_forecast(steps=horizon, exog=exog_block)
        ci = fc.conf_int(alpha=alpha)

        block = pd.DataFrame(
            {
                "forecast": np.asarray(fc.predicted_mean, dtype=float),
                "lower": np.asarray(ci.iloc[:, 0], dtype=float),
                "upper": np.asarray(ci.iloc[:, 1], dtype=float),
                "origin": origin,
                "step": np.arange(1, horizon + 1),
            },
            index=idx,
        )
        blocks.append(block)

        # Advance the state with the now-observed block (never refit).
        previous = current
        current = current.append(
            y_test.iloc[start:stop],
            exog=exog_block,
            refit=False,
        )
        # Each appended results object retains its own smoother output. Without
        # dropping the superseded one the chain holds 14 full state-space
        # results simultaneously, which exhausts memory on modest machines.
        if previous is not res:
            del previous
        del fc, ci
        gc.collect()

    out = pd.concat(blocks)
    out.index.name = "date"
    return out


---
## 10. Foundation model (functions)

**Assignment Part 8.** Runs a genuine Chronos model or reports honestly that it could not. A benchmark is never relabelled as a foundation-model result.

In [ ]:
"""
Time-series foundation model (assignment Part 8).

**Honesty contract for this module**

The demo pipeline this project started from returned a daily seasonal naive
forecast from a function called ``forecast_foundation_model_placeholder`` and
reported it in the comparison table under the name ``foundation_model``. That
is a misattribution: it makes a benchmark look like a foundation model and
would invalidate any conclusion drawn about foundation-model performance.

This module therefore guarantees:

1. :func:`chronos_forecast` runs a *real* Chronos model or raises. It never
   silently substitutes another method.
2. :func:`run_foundation_model` returns a status of ``executed`` or
   ``not_executed``. When it is ``not_executed`` no forecast is produced, no row
   enters the comparison table, and the analysis summary records the reason.
3. Nothing in this module ever labels a benchmark forecast as a foundation-model
   forecast.

Requirements to actually run it
-------------------------------
* Package: ``chronos-forecasting`` (which pulls in ``torch`` and
  ``transformers``); install with ``pip install chronos-forecasting``.
* Weights: downloaded from the Hugging Face Hub on first use
  (``amazon/chronos-bolt-base``, roughly 200 MB). **Network access to
  ``huggingface.co`` is required for that first download only**; afterwards the
  model is cached under ``~/.cache/huggingface`` and runs fully offline.
* Hardware: CPU is sufficient. Chronos-Bolt is a patch-based encoder-decoder
  that produces the whole horizon in a single forward pass, so a 24-step
  forecast per origin takes on the order of a second on CPU.
* API key: none. Chronos runs locally, which is why it is preferred here over
  TimeGPT (see :func:`alternative_foundation_models`).

Reproducibility limitations
---------------------------
Chronos-Bolt is deterministic in its quantile outputs given fixed weights and a
fixed context, so repeated runs agree. Reproducibility is nonetheless tied to a
specific weight revision on the Hub: pin ``CHRONOS_MODEL_ID`` and record the
resolved commit hash (this module does) if exact numbers must be reproduced.
"""


import importlib.util
import time
from typing import Any

import numpy as np
import pandas as pd



# ------------------------------------------------------------------
# Availability
# ------------------------------------------------------------------


def chronos_available() -> dict[str, Any]:
    """
    Report whether Chronos can actually be run in this environment.

    Distinguishes the two failure modes that matter, because they need
    different fixes: the *package* being absent (``pip install``) versus the
    *weights* being unreachable (network access to the Hugging Face Hub).
    """
    has_torch = importlib.util.find_spec("torch") is not None
    has_chronos = importlib.util.find_spec("chronos") is not None

    weights_cached = False
    if has_chronos:
        try:
            from huggingface_hub import try_to_load_from_cache

            hit = try_to_load_from_cache(cfg.CHRONOS_MODEL_ID, "config.json")
            weights_cached = isinstance(hit, str)
        except Exception:  # noqa: BLE001
            weights_cached = False

    return {
        "torch_installed": has_torch,
        "chronos_installed": has_chronos,
        "weights_cached_locally": weights_cached,
        "can_run": bool(has_torch and has_chronos),
        "model_id": cfg.CHRONOS_MODEL_ID,
    }


def alternative_foundation_models() -> dict[str, dict[str, Any]]:
    """Factual comparison of the three candidate foundation models."""
    return {
        "chronos_bolt": {
            "package": "chronos-forecasting",
            "runs_locally": True,
            "api_key_required": False,
            "weights_source": "Hugging Face Hub (amazon/chronos-bolt-base)",
            "approx_download_mb": 200,
            "input": "1-D array of past target values (context), plus horizon",
            "output": "quantile forecasts; median used as the point forecast",
            "offline_after_first_download": True,
            "note": "Preferred: no API key, no per-call cost, fully reproducible offline.",
        },
        "timesfm": {
            "package": "timesfm",
            "runs_locally": True,
            "api_key_required": False,
            "weights_source": "Hugging Face Hub (google/timesfm-*)",
            "approx_download_mb": 800,
            "input": "context array plus frequency indicator",
            "output": "point forecast (quantiles in newer versions)",
            "offline_after_first_download": True,
            "note": "Heavier dependency stack; a viable second choice.",
        },
        "timegpt": {
            "package": "nixtla",
            "runs_locally": False,
            "api_key_required": True,
            "weights_source": "hosted API (weights not distributed)",
            "approx_download_mb": 0,
            "input": "dataframe posted to the Nixtla API",
            "output": "point forecast and prediction intervals",
            "offline_after_first_download": False,
            "note": (
                "Cannot run offline. Results depend on a server-side model that "
                "may change without notice, so exact reproducibility is not "
                "guaranteed. Sending household sensor data to a third-party API "
                "is also a privacy consideration for smart-home deployment."
            ),
        },
    }


# ------------------------------------------------------------------
# Real Chronos inference
# ------------------------------------------------------------------


def load_chronos(model_id: str = cfg.CHRONOS_MODEL_ID) -> Any:
    """
    Load a Chronos pipeline. Raises if the package or weights are unavailable.

    Deliberately allows the exception to propagate: a caller must not be able to
    mistake a failed load for a successful forecast.
    """
    import torch
    from chronos import BaseChronosPipeline

    return BaseChronosPipeline.from_pretrained(
        model_id,
        device_map="cpu",
        torch_dtype=torch.float32,
    )


def chronos_forecast(
    pipeline: Any,
    context: pd.Series,
    horizon: int = cfg.HORIZON,
    context_length: int = cfg.CHRONOS_CONTEXT_LENGTH,
    quantile_levels: tuple[float, ...] = (0.1, 0.5, 0.9),
) -> pd.DataFrame:
    """
    One zero-shot ``horizon``-step forecast from a real Chronos pipeline.

    Only the most recent ``context_length`` observations are supplied, which is
    what the model was trained to consume. The 0.5 quantile is the point
    forecast; the outer quantiles give a prediction interval directly, without
    the Gaussian assumption that SARIMAX's analytic intervals rely on.
    """
    import torch

    values = pd.Series(context).dropna().to_numpy(dtype=float)[-context_length:]
    tensor = torch.tensor(values, dtype=torch.float32).unsqueeze(0)

    quantiles, _mean = pipeline.predict_quantiles(
        context=tensor,
        prediction_length=horizon,
        quantile_levels=list(quantile_levels),
    )
    array = quantiles[0].numpy()

    return pd.DataFrame(
        {f"q{int(q * 100)}": array[:, i] for i, q in enumerate(quantile_levels)}
    )


def rolling_origin_chronos(
    y_full: pd.Series,
    test_index: pd.DatetimeIndex,
    horizon: int = cfg.HORIZON,
    model_id: str = cfg.CHRONOS_MODEL_ID,
) -> dict[str, Any]:
    """
    Rolling-origin Chronos forecasts, matching the other models exactly.

    Zero-shot: the model is never fine-tuned on this dataset. At origin ``i`` it
    receives only observations strictly before the block, so the evaluation
    protocol is identical to the benchmarks, SARIMAX and the ML model.
    """
    pipeline = load_chronos(model_id)
    n_origins = len(test_index) // horizon

    point_blocks: list[pd.Series] = []
    interval_blocks: list[pd.DataFrame] = []
    t0 = time.time()

    for origin in range(n_origins):
        block = test_index[origin * horizon : (origin + 1) * horizon]
        context = y_full.loc[y_full.index < block[0]]

        quantiles = chronos_forecast(pipeline, context, horizon=horizon)
        quantiles.index = block

        point_blocks.append(pd.Series(quantiles["q50"].to_numpy(), index=block))
        interval_blocks.append(
            pd.DataFrame(
                {
                    "forecast": quantiles["q50"].to_numpy(),
                    "lower": quantiles["q10"].to_numpy(),
                    "upper": quantiles["q90"].to_numpy(),
                    "origin": origin,
                    "step": np.arange(1, horizon + 1),
                },
                index=block,
            )
        )

    point = pd.concat(point_blocks)
    point.index.name = "date"
    intervals = pd.concat(interval_blocks)
    intervals.index.name = "date"

    return {
        "forecast": point,
        "intervals": intervals,
        "runtime_seconds": round(time.time() - t0, 2),
        "n_origins": n_origins,
        "model_id": model_id,
        "mode": "zero-shot (no fine-tuning on this dataset)",
    }


# ------------------------------------------------------------------
# Guarded entry point
# ------------------------------------------------------------------


def run_foundation_model(
    y_full: pd.Series,
    test_index: pd.DatetimeIndex,
    horizon: int = cfg.HORIZON,
) -> dict[str, Any]:
    """
    Attempt a genuine foundation-model forecast; report failure honestly.

    Returns a record whose ``status`` is either ``executed`` (``forecast`` holds
    a real Chronos forecast) or ``not_executed`` (``forecast`` is ``None`` and
    ``reason``/``how_to_run`` explain what is missing). **No fallback forecast is
    ever produced.**
    """
    availability = chronos_available()

    if not availability["can_run"]:
        missing = []
        if not availability["chronos_installed"]:
            missing.append("chronos-forecasting")
        if not availability["torch_installed"]:
            missing.append("torch")
        return {
            "status": "not_executed",
            "result_marker": "NOT EXECUTED",
            "forecast": None,
            "intervals": None,
            "availability": availability,
            "reason": f"Required package(s) not installed: {', '.join(missing)}.",
            "how_to_run": how_to_run_locally(),
            "alternatives": alternative_foundation_models(),
        }

    try:
        outcome = rolling_origin_chronos(y_full, test_index, horizon=horizon)
        return {
            "status": "executed",
            "result_marker": "EXECUTED",
            "availability": availability,
            **outcome,
        }
    except Exception as exc:  # noqa: BLE001
        return {
            "status": "not_executed",
            "result_marker": "NOT EXECUTED",
            "forecast": None,
            "intervals": None,
            "availability": availability,
            "reason": (
                f"{type(exc).__name__}: {exc}. The most common cause is that the "
                "model weights could not be downloaded from the Hugging Face Hub "
                "(no network access, or the host is blocked)."
            ),
            "how_to_run": how_to_run_locally(),
            "alternatives": alternative_foundation_models(),
        }


def how_to_run_locally() -> dict[str, Any]:
    """Exact instructions for producing the missing foundation-model results."""
    return {
        "step_1_install": "pip install chronos-forecasting",
        "step_2_network": (
            "Ensure https://huggingface.co is reachable for the first run so "
            f"the weights for {cfg.CHRONOS_MODEL_ID} (~200 MB) can be cached."
        ),
        "step_3_run": "python scripts/run_pipeline.py",
        "step_4_verify": (
            "Check outputs/analysis/model_analysis_summary.json: the "
            "foundation_model block should read status='executed'. If it still "
            "reads 'not_executed', the recorded 'reason' names the cause."
        ),
        "offline_note": (
            "After the first successful download, set HF_HUB_OFFLINE=1 to run "
            "with no network at all."
        ),
        "expected_runtime": (
            "14 origins x 24 steps on CPU: on the order of 10-60 seconds total "
            "for chronos-bolt-base."
        ),
    }


---
## 11. Figures (functions)

**Assignment Parts 1, 2 and 10.** Every figure answers a specific analytical question.

In [ ]:
"""
Figure generation (assignment Parts 1, 2, 10).

Every figure here answers a specific analytical question; none is decorative.
All are saved at ``config.FIG_DPI`` with descriptive titles, labelled axes and
units, and legends wherever more than one series is drawn.

A note on scaling: forecast plots always show the actual observations on the
same axes and at the same scale as the forecasts, and the y-axis is never
truncated to exaggerate differences between models.
"""


from pathlib import Path
from typing import Any, Mapping, Sequence

import matplotlib
import numpy as np
import pandas as pd

import matplotlib.dates as mdates  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402


UNITS = "Appliance energy use (Wh, hourly mean of 10-min readings)"

plt.rcParams.update(
    {
        "figure.autolayout": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)


def _save(fig: plt.Figure, name: str) -> Path:
    """Save a figure to the figures directory and close it."""
    cfg.ensure_dirs()
    path = cfg.FIGURE_DIR / f"{name}.{cfg.FIG_FORMAT}"
    fig.savefig(path, dpi=cfg.FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return path


# ------------------------------------------------------------------
# Part 1: exploratory figures
# ------------------------------------------------------------------


def plot_full_series(y: pd.Series) -> Path:
    """Whole record: shows the overall level, volatility and any regime change."""
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(y.index, y.to_numpy(), linewidth=0.6, color="#1f77b4")
    ax.set_title(
        f"Appliance energy use, full record ({y.index.min():%d %b %Y} to "
        f"{y.index.max():%d %b %Y}, hourly)"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel(UNITS)
    return _save(fig, "01_full_series")


def plot_recent_series(y: pd.Series, days: int = 14) -> Path:
    """Recent window: the daily rhythm is invisible at full-record scale."""
    recent = y.iloc[-days * 24 :]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(recent.index, recent.to_numpy(), linewidth=1.2, color="#1f77b4")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_title(f"Appliance energy use, final {days} days (the test period)")
    ax.set_xlabel("Date")
    ax.set_ylabel(UNITS)
    return _save(fig, "02_recent_series")


def plot_daily_profile(y: pd.Series) -> Path:
    """Mean and interquartile band by hour of day: locates the daily cycle."""
    grouped = y.groupby(y.index.hour)
    mean = grouped.mean()
    q25, q75 = grouped.quantile(0.25), grouped.quantile(0.75)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(mean.index, mean.to_numpy(), marker="o", color="#d62728", label="Mean")
    ax.fill_between(
        mean.index, q25.to_numpy(), q75.to_numpy(),
        alpha=0.25, color="#d62728", label="Interquartile range",
    )
    ax.set_xticks(range(0, 24, 2))
    ax.set_title("Average daily profile of appliance energy use")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel(UNITS)
    ax.legend()
    return _save(fig, "03_daily_profile")


def plot_weekly_profile(y: pd.Series) -> Path:
    """Mean profile across the week: tests whether weekdays differ from weekends."""
    frame = pd.DataFrame({"y": y.to_numpy()}, index=y.index)
    frame["slot"] = frame.index.dayofweek * 24 + frame.index.hour
    profile = frame.groupby("slot")["y"].mean()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(profile.index, profile.to_numpy(), color="#2ca02c", linewidth=1.5)
    for boundary in range(0, 169, 24):
        ax.axvline(boundary, color="grey", linewidth=0.6, linestyle=":")
    ax.set_xticks([12 + 24 * i for i in range(7)])
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    ax.set_xlim(0, 168)
    ax.set_title("Average weekly profile (mean by hour-of-week, 168 slots)")
    ax.set_xlabel("Day of week")
    ax.set_ylabel(UNITS)
    return _save(fig, "04_weekly_profile")


def plot_distribution(y: pd.Series) -> Path:
    """Histogram on linear and log scales: quantifies the right skew."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].hist(y.to_numpy(), bins=60, color="#1f77b4", edgecolor="white")
    axes[0].axvline(float(y.mean()), color="red", linestyle="--", label=f"Mean {y.mean():.0f}")
    axes[0].axvline(float(y.median()), color="black", linestyle="-.", label=f"Median {y.median():.0f}")
    axes[0].set_title("Distribution of hourly appliance energy use")
    axes[0].set_xlabel(UNITS)
    axes[0].set_ylabel("Frequency (hours)")
    axes[0].legend()

    axes[1].hist(np.log(y.to_numpy()), bins=60, color="#ff7f0e", edgecolor="white")
    axes[1].set_title("Same data on a log scale (skew made visible)")
    axes[1].set_xlabel("log(Wh)")
    axes[1].set_ylabel("Frequency (hours)")

    fig.tight_layout()
    return _save(fig, "05_distribution")


def plot_boxplots_by_hour(y: pd.Series) -> Path:
    """Hourly boxplots: shows that both level and spread vary with time of day."""
    data = [y[y.index.hour == h].to_numpy() for h in range(24)]
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.boxplot(data, tick_labels=[str(h) for h in range(24)], showfliers=True,
               flierprops={"markersize": 2, "alpha": 0.3})
    ax.set_title("Appliance energy use by hour of day (heteroskedastic: spread grows in the evening)")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel(UNITS)
    return _save(fig, "06_boxplot_by_hour")


def plot_boxplots_by_dayofweek(y: pd.Series) -> Path:
    """Day-of-week boxplots: tests whether a weekly term is warranted."""
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    data = [y[y.index.dayofweek == d].to_numpy() for d in range(7)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.boxplot(data, tick_labels=labels, showfliers=True,
               flierprops={"markersize": 2, "alpha": 0.3})
    ax.set_title("Appliance energy use by day of week")
    ax.set_xlabel("Day of week")
    ax.set_ylabel(UNITS)
    return _save(fig, "07_boxplot_by_dayofweek")


# ------------------------------------------------------------------
# Part 2: time-series structure
# ------------------------------------------------------------------


def plot_decomposition(result: Any, period: int = cfg.DAILY_PERIOD, days: int = 21) -> Path:
    """STL components over a readable window (the full record is unreadable)."""
    observed = pd.Series(result.observed).iloc[-days * 24 :]
    trend = pd.Series(result.trend).iloc[-days * 24 :]
    seasonal = pd.Series(result.seasonal).iloc[-days * 24 :]
    resid = pd.Series(result.resid).iloc[-days * 24 :]

    fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
    for ax, series, label, colour in zip(
        axes,
        [observed, trend, seasonal, resid],
        ["Observed", "Trend", f"Seasonal (period={period}h)", "Remainder"],
        ["#1f77b4", "#d62728", "#2ca02c", "#7f7f7f"],
    ):
        ax.plot(series.index, series.to_numpy(), linewidth=1.0, color=colour)
        ax.set_ylabel(label, fontsize=9)
    axes[0].set_title(f"STL decomposition (robust), final {days} days, period = {period} hours")
    axes[-1].set_xlabel("Date")
    fig.tight_layout()
    return _save(fig, "08_stl_decomposition")


def plot_acf_pacf(y: pd.Series, nlags: int = 168) -> Path:
    """
    ACF and PACF to lag 168.

    Spikes at 24, 48, 72 identify the daily cycle; a spike at 168 identifies the
    weekly one. This is the evidence for the choice of seasonal period.
    """
    acf_vals, pacf_vals, band = acf_pacf_values(y, nlags=nlags)

    fig, axes = plt.subplots(2, 1, figsize=(13, 7))
    for ax, values, label in (
        (axes[0], acf_vals, "Autocorrelation (ACF)"),
        (axes[1], pacf_vals, "Partial autocorrelation (PACF)"),
    ):
        lags = np.arange(len(values))
        ax.vlines(lags, 0, values, color="#1f77b4", linewidth=1.0)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.axhline(band, color="red", linestyle="--", linewidth=0.8, label="95% white-noise band")
        ax.axhline(-band, color="red", linestyle="--", linewidth=0.8)
        for multiple in range(24, len(values), 24):
            ax.axvline(multiple, color="green", linestyle=":", alpha=0.5)
        ax.set_ylabel(label)
        ax.set_xlabel("Lag (hours)")
        ax.legend(loc="upper right")
    axes[0].set_title(
        "ACF and PACF of hourly appliance energy use "
        "(green dotted lines mark multiples of 24 h)"
    )
    fig.tight_layout()
    return _save(fig, "09_acf_pacf")


def plot_stationarity_diagnostics(y: pd.Series, window: int = 168) -> Path:
    """
    Rolling mean and standard deviation, plus the differenced series.

    A visual companion to ADF/KPSS: drift in the rolling mean indicates a
    non-constant level, drift in the rolling standard deviation indicates
    non-constant variance.
    """
    rolling_mean = y.rolling(window).mean()
    rolling_std = y.rolling(window).std()
    differenced = y.diff().dropna()

    fig, axes = plt.subplots(2, 1, figsize=(13, 8))
    axes[0].plot(y.index, y.to_numpy(), linewidth=0.5, alpha=0.4, color="grey", label="Observed")
    axes[0].plot(rolling_mean.index, rolling_mean.to_numpy(), color="#d62728",
                 linewidth=1.8, label=f"Rolling mean ({window} h)")
    axes[0].plot(rolling_std.index, rolling_std.to_numpy(), color="#2ca02c",
                 linewidth=1.8, label=f"Rolling std ({window} h)")
    axes[0].set_title(f"Rolling mean and standard deviation ({window}-hour window)")
    axes[0].set_ylabel(UNITS)
    axes[0].legend()

    axes[1].plot(differenced.index, differenced.to_numpy(), linewidth=0.5, color="#1f77b4")
    axes[1].set_title("First difference of the series")
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Change in Wh")
    fig.tight_layout()
    return _save(fig, "10_stationarity_diagnostics")


# ------------------------------------------------------------------
# Parts 5 and 10: forecast figures
# ------------------------------------------------------------------


def _context_and_test(ax, train: pd.Series, test: pd.Series, context_days: int = 3) -> None:
    """Draw shared context: recent training data plus the observed test period."""
    context = train.iloc[-context_days * 24 :]
    ax.plot(context.index, context.to_numpy(), color="grey", linewidth=1.0,
            alpha=0.7, label="Training data (context)")
    ax.plot(test.index, test.to_numpy(), color="black", linewidth=1.8, label="Actual (test)")
    ax.axvline(test.index[0], color="red", linestyle="--", linewidth=1.2,
               label="Start of test period")


def plot_benchmark_forecasts(
    train: pd.Series, test: pd.Series, forecasts: Mapping[str, pd.Series]
) -> Path:
    """All five benchmarks against the actual test series."""
    fig, ax = plt.subplots(figsize=(15, 6))
    _context_and_test(ax, train, test)
    for name, series in forecasts.items():
        ax.plot(series.index, series.to_numpy(), linewidth=1.2, alpha=0.85, label=name)
    ax.set_title(
        f"Benchmark forecasts, rolling origin ({cfg.HORIZON}-hour horizon, "
        f"{cfg.N_ORIGINS} origins over {cfg.TEST_DAYS} days)"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel(UNITS)
    ax.legend(ncol=2, fontsize=9)
    return _save(fig, "11_benchmark_forecasts")


def plot_forecast_with_intervals(
    train: pd.Series,
    test: pd.Series,
    intervals: pd.DataFrame,
    name: str,
    filename: str,
    interval_label: str = "95% prediction interval",
) -> Path:
    """Point forecast plus prediction interval for one model."""
    fig, ax = plt.subplots(figsize=(15, 6))
    _context_and_test(ax, train, test)
    ax.plot(intervals.index, intervals["forecast"].to_numpy(),
            color="#ff7f0e", linewidth=1.6, label=f"{name} forecast")
    ax.fill_between(
        intervals.index,
        intervals["lower"].to_numpy(),
        intervals["upper"].to_numpy(),
        color="#ff7f0e", alpha=0.2, label=interval_label,
    )
    ax.set_title(f"{name}: {cfg.HORIZON}-hour rolling-origin forecasts with uncertainty")
    ax.set_xlabel("Date")
    ax.set_ylabel(UNITS)
    ax.legend()
    return _save(fig, filename)


def plot_single_forecast(
    train: pd.Series, test: pd.Series, forecast: pd.Series, name: str, filename: str
) -> Path:
    """Point forecast for one model against the actuals."""
    fig, ax = plt.subplots(figsize=(15, 6))
    _context_and_test(ax, train, test)
    ax.plot(forecast.index, forecast.to_numpy(), color="#9467bd",
            linewidth=1.6, label=f"{name} forecast")
    ax.set_title(f"{name}: {cfg.HORIZON}-hour rolling-origin forecasts")
    ax.set_xlabel("Date")
    ax.set_ylabel(UNITS)
    ax.legend()
    return _save(fig, filename)


def plot_forecast_comparison(
    train: pd.Series,
    test: pd.Series,
    forecasts: Mapping[str, pd.Series],
    highlight: Sequence[str] | None = None,
    zoom_days: int = 4,
) -> Path:
    """
    Final comparison: the whole test period plus a zoomed window.

    The zoom exists because 336 hours of several overlapping forecasts is
    unreadable; both panels use the same y-scale so the zoom cannot mislead.
    """
    highlight = list(highlight or forecasts.keys())
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    for ax, window in ((axes[0], None), (axes[1], zoom_days * 24)):
        subset_test = test if window is None else test.iloc[:window]
        _context_and_test(ax, train, subset_test, context_days=2)
        for name in highlight:
            series = forecasts[name]
            series = series if window is None else series.iloc[:window]
            ax.plot(series.index, series.to_numpy(), linewidth=1.3, alpha=0.9, label=name)
        ax.set_ylabel(UNITS)
        ax.legend(ncol=3, fontsize=8)

    axes[0].set_title(
        f"Forecast comparison over the full {cfg.TEST_DAYS}-day test period "
        f"({cfg.HORIZON}-hour rolling-origin forecasts)"
    )
    axes[1].set_title(f"Same forecasts, first {zoom_days} days (detail)")
    axes[1].set_xlabel("Date")
    fig.tight_layout()
    return _save(fig, "15_forecast_comparison")


# ------------------------------------------------------------------
# Diagnostics and error figures
# ------------------------------------------------------------------


def plot_residual_diagnostics(residuals: pd.Series, model_name: str = "SARIMAX") -> Path:
    """
    Four-panel residual check: time plot, ACF, histogram and Q-Q plot.

    Time plot -> changing variance; ACF -> leftover autocorrelation;
    histogram and Q-Q -> the normality assumption behind the intervals.
    """
    from scipy import stats

    resid = pd.Series(residuals).dropna()
    acf_vals, _, band = acf_pacf_values(resid, nlags=48)

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    axes[0, 0].plot(resid.index, resid.to_numpy(), linewidth=0.6, color="#1f77b4")
    axes[0, 0].axhline(0, color="red", linewidth=0.8)
    axes[0, 0].set_title("Residuals over time")
    axes[0, 0].set_ylabel("Residual (Wh)")

    lags = np.arange(len(acf_vals))
    axes[0, 1].vlines(lags, 0, acf_vals, color="#1f77b4")
    axes[0, 1].axhline(band, color="red", linestyle="--", linewidth=0.8)
    axes[0, 1].axhline(-band, color="red", linestyle="--", linewidth=0.8)
    axes[0, 1].axhline(0, color="black", linewidth=0.8)
    axes[0, 1].set_title("Residual ACF (lags 0-48)")
    axes[0, 1].set_xlabel("Lag (hours)")

    axes[1, 0].hist(resid.to_numpy(), bins=50, color="#2ca02c", edgecolor="white", density=True)
    grid = np.linspace(resid.min(), resid.max(), 200)
    axes[1, 0].plot(grid, stats.norm.pdf(grid, resid.mean(), resid.std()),
                    color="red", linewidth=1.5, label="Normal fit")
    axes[1, 0].set_title("Residual distribution")
    axes[1, 0].set_xlabel("Residual (Wh)")
    axes[1, 0].legend()

    stats.probplot(resid.to_numpy(), dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title("Normal Q-Q plot")

    fig.suptitle(f"{model_name} residual diagnostics", fontsize=13)
    fig.tight_layout()
    return _save(fig, "16_residual_diagnostics")


def plot_error_comparison(comparison: pd.DataFrame, metrics: Sequence[str] = ("MASE", "RMSE", "MAE")) -> Path:
    """Ranked bar charts of the headline metrics, lower being better."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 6), sharey=True)
    axes = np.atleast_1d(axes)

    ordered = comparison.sort_values(metrics[0], ascending=False)
    for ax, metric in zip(axes, metrics):
        colours = ["#2ca02c" if v == ordered[metric].min() else "#1f77b4"
                   for v in ordered[metric]]
        ax.barh(ordered["model"], ordered[metric].to_numpy(), color=colours)
        ax.set_xlabel(metric)
        ax.set_title(f"{metric} (lower is better)")
        if metric == "MASE":
            ax.axvline(1.0, color="red", linestyle="--", linewidth=1.0,
                       label="MASE = 1 (seasonal naive)")
            ax.legend(fontsize=8)
    fig.suptitle("Model error comparison on the common test sample", fontsize=13)
    fig.tight_layout()
    return _save(fig, "17_error_comparison")


def plot_error_by_horizon(errors: pd.DataFrame, metric: str = "RMSE") -> Path:
    """Error as a function of lead time: shows how fast each model degrades."""
    fig, ax = plt.subplots(figsize=(11, 6))
    for name, group in errors.groupby("model"):
        ordered = group.sort_values("step")
        ax.plot(ordered["step"], ordered[metric], marker="o", markersize=3, label=name)
    ax.set_title(f"{metric} by forecast lead time (1-{cfg.HORIZON} hours ahead)")
    ax.set_xlabel("Hours ahead of the forecast origin")
    ax.set_ylabel(metric)
    ax.legend(ncol=2, fontsize=8)
    return _save(fig, "18_error_by_horizon")


def plot_feature_importance(importance: pd.DataFrame, top_n: int = 20) -> Path:
    """Top permutation importances, measured on held-out training data."""
    top = importance.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(top["feature"], top["importance_mean"],
            xerr=top["importance_std"], color="#1f77b4")
    ax.set_xlabel("Increase in RMSE when the feature is permuted (Wh)")
    ax.set_title(f"Top {top_n} features by permutation importance\n(held-out tail of the training period)")
    fig.tight_layout()
    return _save(fig, "19_feature_importance")


def plot_feature_ablation(ablation: pd.DataFrame, metric: str = "RMSE") -> Path:
    """Cumulative feature-group ablation: the marginal value of each group."""
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(ablation["feature_set"], ablation[metric], marker="o", linewidth=2, color="#d62728")
    for x, value in zip(ablation["feature_set"], ablation[metric]):
        ax.annotate(f"{value:.1f}", (x, value), textcoords="offset points",
                    xytext=(0, 9), ha="center", fontsize=9)
    ax.set_title(f"Feature-group ablation: test {metric} as groups are added")
    ax.set_ylabel(metric)
    ax.set_xlabel("Cumulative feature set")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    fig.tight_layout()
    return _save(fig, "20_feature_ablation")


---
## 12. Analysis outputs (functions)

**Assignment Parts 11–12.** Writes facts and questions, never interpretation.

In [ ]:
"""
Machine-readable analysis outputs (assignment Parts 11, 12 and the leakage audit).

Report-assistance boundary
--------------------------
This module writes **facts**: parameters, metrics, test statistics, rankings,
runtimes and explicit flags. It also writes *questions* for the analyst to
answer. It deliberately does **not** write interpretive prose, conclusions or
report paragraphs. Where a value invites interpretation, the output states the
value and the comparison, and leaves the judgement to the analyst.
"""


import json
import platform
import sys
from datetime import datetime, timezone
from typing import Any, Mapping

import numpy as np
import pandas as pd



# ------------------------------------------------------------------
# Serialisation helpers
# ------------------------------------------------------------------


def _jsonable(obj: Any) -> Any:
    """Convert numpy/pandas objects into JSON-serialisable Python types."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        value = float(obj)
        return None if not np.isfinite(value) else value
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, (np.ndarray,)):
        return [_jsonable(v) for v in obj.tolist()]
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if isinstance(obj, pd.DataFrame):
        return [_jsonable(r) for r in obj.to_dict(orient="records")]
    if isinstance(obj, pd.Series):
        return _jsonable(obj.to_dict())
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    return obj


def environment_record() -> dict[str, Any]:
    """Package versions and platform, for reproducibility."""
    versions: dict[str, str] = {}
    for name in ("numpy", "pandas", "scipy", "sklearn", "statsmodels", "matplotlib",
                 "xgboost", "lightgbm", "torch", "chronos"):
        try:
            module = __import__(name)
            versions[name] = getattr(module, "__version__", "unknown")
        except ImportError:
            versions[name] = "not installed"

    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "packages": versions,
        "random_state": cfg.RANDOM_STATE,
        "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


# ------------------------------------------------------------------
# Analysis summary
# ------------------------------------------------------------------


def build_analysis_summary(
    *,
    quality: Mapping[str, Any],
    aggregation: Mapping[str, Any],
    eda: Mapping[str, Any],
    stationarity: Mapping[str, Any],
    experiment: Mapping[str, Any],
    sarimax: Mapping[str, Any],
    ml: Mapping[str, Any],
    foundation: Mapping[str, Any],
    comparison: pd.DataFrame,
    dm_tests: pd.DataFrame,
    runtimes: Mapping[str, float],
) -> dict[str, Any]:
    """Collect every factual result into one nested record."""
    ranking = comparison[["rank", "model", "MAE", "RMSE", "MASE", "Bias", "sMAPE", "n"]]

    benchmark_names = [
        "mean", "naive", "seasonal_naive_daily", "seasonal_naive_weekly", "drift",
    ]
    benchmark_rows = comparison[comparison["model"].isin(benchmark_names)]

    return {
        "project": "Appliance energy forecasting (UCI energydata_complete)",
        "environment": environment_record(),
        "data_quality": _jsonable(quality),
        "aggregation": _jsonable(aggregation),
        "eda": _jsonable(eda),
        "stationarity_and_seasonality": _jsonable(stationarity),
        "experiment_design": _jsonable(experiment),
        "sarimax": _jsonable(sarimax),
        "machine_learning_model": _jsonable(ml),
        "foundation_model": _jsonable(foundation),
        "model_ranking": _jsonable(ranking),
        "benchmark_ranking": _jsonable(
            benchmark_rows.sort_values(cfg.PRIMARY_METRIC)[
                ["model", "MAE", "RMSE", "MASE", "Bias"]
            ]
        ),
        "strongest_benchmark": (
            str(benchmark_rows.sort_values(cfg.PRIMARY_METRIC).iloc[0]["model"])
            if not benchmark_rows.empty else None
        ),
        "best_overall_model": str(comparison.iloc[0]["model"]),
        "diebold_mariano_tests": _jsonable(dm_tests),
        "runtimes_seconds": _jsonable(runtimes),
        "analytical_questions": analytical_prompts(),
    }


def analytical_prompts() -> dict[str, list[str]]:
    """
    Questions for the analyst, mapped to the six assignment questions.

    These are prompts, not answers: each points at a specific number in this
    file and asks what it implies.
    """
    return {
        "Q1_benchmarks": [
            "Which benchmark has the lowest MASE, and by what margin over the second best?",
            "Compare seasonal_naive_daily against seasonal_naive_weekly: what does the "
            "winner imply about the relative strength of the 24-hour and 168-hour cycles?",
            "The mean forecast ignores all dynamics and naive ignores all seasonality. "
            "Where do they rank, and what does that ordering say about how much of the "
            "variance is seasonal rather than persistent?",
            "Check the Bias column: which benchmarks systematically over- or "
            "under-forecast, and can you explain that from the daily profile?",
        ],
        "Q2_sarimax": [
            "Compare SARIMAX MASE/RMSE with the strongest benchmark. Is the difference "
            "significant according to the Diebold-Mariano test at the 5% level?",
            "The selected seasonal order includes D and Q terms at s=24. What does that "
            "say about how the daily cycle is being modelled?",
            "Seasonal strength Fs is reported in stationarity_and_seasonality, and AIC "
            "selected a different D than the Fs rule of thumb recommends. How do you "
            "reconcile the two pieces of evidence?",
            "Does the Ljung-Box p-value at lag 24 indicate remaining autocorrelation? "
            "If so, what structure might the model still be missing?",
            "Compare the exogenous and target-only SARIMAX variants. How much does "
            "adding weather regressors change accuracy, and is that gain conditional?",
        ],
        "Q3_feature_model": [
            "In the ablation table, which cumulative feature group produces the largest "
            "single drop in RMSE?",
            "In the grouped permutation importance, which group carries the most weight, "
            "and does it agree with the ablation?",
            "Do rolling-window features add anything once lags are already present, or is "
            "the information redundant?",
            "Do indoor sensor variables add measurable value beyond weather variables?",
        ],
        "Q4_foundation_model": [
            "Check foundation_model.status. If it is 'not_executed', no claim about "
            "foundation-model accuracy can be made from this run -- state that plainly.",
            "If executed: quantify the difference against the strongest benchmark, "
            "SARIMAX and the ML model in both MASE and RMSE, and report the "
            "Diebold-Mariano p-values rather than the raw gaps alone.",
            "The model is zero-shot. Is it reasonable to expect it to beat a model "
            "trained on 4 months of this specific household's data?",
        ],
        "Q5_covariate_availability": [
            "Using the leakage audit, list which variables are genuinely known at the "
            "forecast origin, which would need their own forecast, and which are "
            "unavailable.",
            "Compare the conditional and operational ML variants. The gap is the value "
            "of perfect weather foresight -- how large is it, and does it change the "
            "model ranking?",
            "The SARIMAX exogenous variant consumes test-period weather. Under what "
            "operational assumption would that be legitimate?",
            "Indoor sensor readings (T1-T9, RH_1-RH_9) and 'lights' are outputs of the "
            "same household process as the target. What does that imply about using "
            "their contemporaneous values as predictors?",
        ],
        "Q6_practical_choice": [
            "Rank the models by MASE, then re-rank by runtime. Does the ordering change?",
            "Which models supply calibrated uncertainty, and which give point forecasts "
            "only? What does that mean for a smart-home controller making decisions?",
            "Which models need refitting as new data arrives, and at what cost?",
            "Given the accuracy differences you measured, is the most accurate model "
            "worth its computational and operational overhead over the simplest "
            "competitive alternative?",
        ],
    }


def write_analysis_summary(summary: Mapping[str, Any]) -> tuple[Any, Any]:
    """Write the summary as both JSON and a readable Markdown digest."""
    cfg.ensure_dirs()

    json_path = cfg.ANALYSIS_DIR / "model_analysis_summary.json"
    json_path.write_text(json.dumps(_jsonable(summary), indent=2))

    md_path = cfg.ANALYSIS_DIR / "model_analysis_summary.md"
    md_path.write_text(_summary_markdown(summary))
    return json_path, md_path


def _summary_markdown(summary: Mapping[str, Any]) -> str:
    """Render the key facts as Markdown tables (no interpretation)."""
    lines: list[str] = [
        "# Model analysis summary (factual outputs only)",
        "",
        "Generated by `scripts/run_pipeline.py`. Every number here was produced by "
        "an executed run; nothing is hand-entered. This file deliberately contains "
        "**no interpretation** -- see `analytical_questions` at the end.",
        "",
    ]

    exp = summary.get("experiment_design", {})
    lines += [
        "## Experiment design",
        "",
        f"- Target: `{exp.get('target')}` ({exp.get('units')})",
        f"- Frequency: {exp.get('frequency')}",
        f"- Training period: {exp.get('train_start')} to {exp.get('train_end')} "
        f"({exp.get('n_train')} observations)",
        f"- Test period: {exp.get('test_start')} to {exp.get('test_end')} "
        f"({exp.get('n_test')} observations)",
        f"- Forecast horizon: {exp.get('horizon')} hours",
        f"- Rolling origins: {exp.get('n_origins')}",
        f"- Primary metric: {exp.get('primary_metric')}",
        "",
    ]

    ranking = summary.get("model_ranking", [])
    if ranking:
        lines += ["## Model ranking", "",
                  "| Rank | Model | MAE | RMSE | MASE | Bias | sMAPE | n |",
                  "|---:|---|---:|---:|---:|---:|---:|---:|"]
        for row in ranking:
            lines.append(
                f"| {row.get('rank')} | {row.get('model')} | "
                f"{_fmt(row.get('MAE'))} | {_fmt(row.get('RMSE'))} | "
                f"{_fmt(row.get('MASE'))} | {_fmt(row.get('Bias'))} | "
                f"{_fmt(row.get('sMAPE'))} | {row.get('n')} |"
            )
        lines.append("")

    sarimax = summary.get("sarimax", {})
    if sarimax:
        lines += [
            "## SARIMAX selection",
            "",
            f"- Selected order: `{sarimax.get('selected_order')}` "
            f"`{sarimax.get('selected_seasonal_order')}`",
            f"- AIC: {_fmt(sarimax.get('selected_aic'))}, BIC: {_fmt(sarimax.get('selected_bic'))}",
            f"- Models attempted: {sarimax.get('n_models_attempted')} "
            f"(converged {sarimax.get('n_models_converged')}, "
            f"failed {sarimax.get('n_models_failed')})",
            f"- Selected exogenous variables: {sarimax.get('selected_exog')}",
            "",
        ]

    foundation = summary.get("foundation_model", {})
    lines += [
        "## Foundation model",
        "",
        f"- Status: **{foundation.get('status', 'unknown').upper()}** "
        f"({foundation.get('result_marker', '')})",
    ]
    if foundation.get("status") != "executed":
        lines.append(f"- Reason: {foundation.get('reason')}")
        lines.append("- No foundation-model row appears in the ranking above, and no "
                     "conclusion about foundation-model accuracy is supported by this run.")
    lines.append("")

    lines += ["## Analytical questions", ""]
    for section, questions in summary.get("analytical_questions", {}).items():
        lines.append(f"### {section}")
        lines += [f"{i}. {q}" for i, q in enumerate(questions, start=1)]
        lines.append("")

    return "\n".join(lines)


def _fmt(value: Any) -> str:
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "n/a"
    if isinstance(value, (int, float, np.floating, np.integer)):
        return f"{float(value):.3f}"
    return str(value)


# ------------------------------------------------------------------
# Leakage audit
# ------------------------------------------------------------------


def build_leakage_audit(
    *,
    aggregation: Mapping[str, Any],
    interpolation_note: str,
    feature_selftest: Mapping[str, Any],
    experiment: Mapping[str, Any],
    exog_selection: Mapping[str, Any],
    ml_regimes: Mapping[str, Any],
    foundation: Mapping[str, Any],
) -> str:
    """
    Produce the honest leakage audit required by the assignment.

    Every row states the component, whether leakage is present, the concrete
    evidence, and the corrective action taken. Rows where leakage is *present
    by design* (the conditional variants) say so rather than hiding it.
    """
    conditional_leak = (
        "YES - by design, labelled" if ml_regimes.get("conditional_included") else "N/A"
    )

    rows = [
        (
            "1. Train/test split",
            "No",
            f"Strictly chronological: train ends {experiment.get('train_end')}, test "
            f"begins {experiment.get('test_start')}. No shuffling anywhere in the "
            "codebase; `train_test_split_chronological` slices by position only.",
            "None required.",
        ),
        (
            "2. Scaling / transformation",
            "No",
            "No scaler is fitted. Gradient-boosted trees are invariant to monotone "
            "feature scaling and SARIMAX is estimated on the raw series, so there is "
            "no scaler that could be fitted on the full sample.",
            "None required.",
        ),
        (
            "3. Hourly aggregation",
            "No",
            "Resampling averages the six 10-minute readings *within* each hour. No "
            "information crosses an hour boundary. "
            f"Partial hours dropped: {aggregation.get('n_partial_hours_dropped')}.",
            "Partial hours dropped rather than imputed.",
        ),
        (
            "4. Missing-value imputation",
            "No",
            interpolation_note,
            "Time interpolation is implemented but was not triggered on this dataset.",
        ),
        (
            "5. Target lag features",
            "No",
            "`shift(lag)` with lag >= 1 for every target lag. Verified programmatically: "
            "perturbing y_t leaves row t unchanged "
            f"(leaking columns: {feature_selftest.get('operational', {}).get('leaking_columns')}).",
            "None required; the self-test runs on every pipeline execution.",
        ),
        (
            "6. Rolling-window features",
            "No",
            "Rolling statistics are computed on `y.shift(1)` *before* `.rolling(w)`, so "
            "the window at row t spans y_{t-w}..y_{t-1}. Covered by the same "
            "perturbation self-test.",
            "None required.",
        ),
        (
            "7. Multi-step forecast construction",
            "No (operational variant)",
            "Recursive forecasting: within a 24-hour block, short lags are filled with "
            "the model's own predictions, never with observed future values. Observed "
            "test data is used only to reset the working series at each new origin, "
            "which represents data genuinely available at that origin.",
            "The original demo pipeline scored the ML model on observed future lags; "
            "that path was removed and replaced with `recursive_forecast`.",
        ),
        (
            "8. Future weather variables (ML, conditional variant)",
            conditional_leak,
            "The `conditional` regime reads contemporaneous weather and sensor values "
            "at time t, which no operator possesses at a 24-hour origin. It is retained "
            "only as an upper bound on the value of perfect covariate foresight.",
            "Reported under an explicit `_conditional` suffix and excluded from any "
            "claim about operational accuracy. The `operational` variant uses lags "
            f">= {cfg.HORIZON} h only.",
        ),
        (
            "9. Future sensor variables",
            conditional_leak,
            "Indoor sensors (T1-T9, RH_1-RH_9) and `lights` are outputs of the same "
            "household process as the target, so their contemporaneous values are "
            "arguably more informative than weather and correspondingly less available.",
            "Same treatment as row 8; in the operational regime they enter only at "
            f"lags of {cfg.EXOG_LAGS} hours.",
        ),
        (
            "10. SARIMAX exogenous regressors",
            "YES - by design, labelled",
            "`get_forecast(exog=X_test)` supplies test-period weather. This is a "
            "CONDITIONAL forecast that assumes future weather is known exactly. "
            f"Selected regressors: {exog_selection.get('selected')}.",
            "A target-only SARIMAX is fitted and reported alongside it; the exogenous "
            "variant carries a `_conditional` label. The pair quantifies the "
            "assumption's value.",
        ),
        (
            "11. SARIMAX parameter estimation",
            "No",
            "Coefficients are estimated on training data only. Rolling-origin "
            "forecasting advances the state with `append(..., refit=False)`, which "
            "filters through newly observed data without re-estimating parameters.",
            "None required.",
        ),
        (
            "12. Order selection (AIC grid search)",
            "No",
            "All 168 candidate models were fitted on the training series only; the "
            "test period was never touched during selection.",
            "None required.",
        ),
        (
            "13. Exogenous variable selection",
            "No",
            "Correlation and VIF screens are computed on the training period only "
            f"(threshold |r| >= {exog_selection.get('corr_threshold')}, "
            f"VIF <= {exog_selection.get('vif_threshold')}).",
            "None required.",
        ),
        (
            "14. Hyper-parameter tuning",
            "No",
            f"`TimeSeriesSplit` with {cfg.ML_CV_SPLITS} expanding-window folds and a "
            f"{cfg.HORIZON - 1}-hour gap, run on the training period only. No random "
            "K-fold anywhere. Grid limited to "
            f"{len(cfg.ML_PARAM_GRID)} candidates to avoid tuning against the test set.",
            "None required.",
        ),
        (
            "15. Feature importance",
            "No",
            "Permutation importance is measured on a held-out tail of the *training* "
            "period, not on the test set.",
            "None required.",
        ),
        (
            "16. Foundation model inputs",
            "No",
            f"Status: {foundation.get('status')}. Chronos is zero-shot and receives only "
            "the context strictly preceding each origin, matching the other models' "
            "protocol.",
            "If not executed, no forecast is produced and no row enters the comparison "
            "table -- a benchmark is never substituted for it.",
        ),
    ]

    lines = [
        "# Data leakage audit",
        "",
        "Every component of the pipeline that could move future information into the "
        "past, audited honestly. Two rows record leakage that is **present by design**: "
        "these are conditional forecasts retained as an upper bound, always labelled, "
        "and never used to support a claim about operational accuracy.",
        "",
        "| Component | Potential leakage? | Evidence | Corrective action |",
        "|---|---|---|---|",
    ]
    for component, verdict, evidence, action in rows:
        lines.append(f"| {component} | {verdict} | {evidence} | {action} |")

    lines += [
        "",
        "## Automated checks that run on every execution",
        "",
        "- `src.features.leakage_self_test` perturbs a single target observation and "
        "asserts that the corresponding feature row is unchanged. A non-empty "
        "`leaking_columns` list fails the audit.",
        "- `src.ml_model.recursive_forecast` raises if any feature row contains NaN, "
        "which would indicate that a value expected to be observed is in fact missing.",
        "- Rolling-origin helpers slice history with `index < block[0]`, a strict "
        "inequality, so the block being forecast is never visible.",
        "",
        "## Residual risks not eliminated",
        "",
        "1. **Single split.** All models are compared on one 14-day window. Metric "
        "differences of a few percent should not be over-read; the Diebold-Mariano "
        "tests in the analysis summary address this directly.",
        "2. **Design choices informed by full-sample EDA.** Lag and window lengths were "
        "chosen from domain reasoning (24 h, 168 h) rather than tuned on the test set, "
        "but the exploratory plots were produced on the whole series.",
        "3. **Conditional variants.** Their accuracy is not achievable operationally "
        "without a weather forecast, whose own error would propagate.",
    ]
    return "\n".join(lines)


def write_leakage_audit(content: str) -> Any:
    """Write the leakage audit to ``outputs/analysis/leakage_audit.md``."""
    cfg.ensure_dirs()
    path = cfg.ANALYSIS_DIR / "leakage_audit.md"
    path.write_text(content)
    return path


---
# EXECUTION

Everything above defines functions. Everything below runs the analysis.

## A. Load the data

**Part 1, steps 1–11.**

In [ ]:
data, quality, aggregation = build_dataset()
data.head()

In [ ]:
q = quality.to_dict()
print("=== DATA QUALITY ===")
for key in ["n_rows", "n_cols", "start", "end", "missing_total", "duplicated_timestamps",
            "duplicated_rows", "inferred_step_minutes", "irregular_steps", "missing_timestamps"]:
    print(f"  {key:25s} {q[key]}")

print("\n=== TARGET ANOMALIES ===")
for key in ["target_min", "target_max", "target_zero_count",
            "target_negative_count", "target_outlier_count_iqr"]:
    print(f"  {key:25s} {q[key]}")

print("\n=== NOTES ===")
for note in quality.notes:
    print(" -", note)

In [ ]:
print("=== HOURLY AGGREGATION ===")
for key, value in aggregation.items():
    if key != "partial_hours_dropped":
        print(f"  {key:35s} {value}")
print()
print(check_interpolation_safety(aggregation))

## B. Exploratory analysis

**Part 1, steps 12–13.**

In [ ]:
y = data[cfg.TARGET]
eda_record = run_eda(data)
pd.Series(eda_record["summary_statistics"])

In [ ]:
pd.Series(eda_record["pattern_evidence"])

In [ ]:
pd.DataFrame(eda_record["top_correlations_with_target"])

In [ ]:
figures = {}
figures["full_series"]  = plot_full_series(y)
figures["recent"]       = plot_recent_series(y)
figures["daily"]        = plot_daily_profile(y)
figures["weekly"]       = plot_weekly_profile(y)
figures["distribution"] = plot_distribution(y)
figures["box_hour"]     = plot_boxplots_by_hour(y)
figures["box_dow"]      = plot_boxplots_by_dayofweek(y)

from IPython.display import Image, display
for key in ["full_series", "daily", "box_hour", "distribution"]:
    display(Image(filename=str(figures[key])))

## C. Stationarity and seasonality

**Part 2.** ADF null = unit root (reject → stationary). KPSS null = stationary
(reject → non-stationary). They are only conclusive together.

In [ ]:
y_train, y_test = train_test_split_chronological(y)
exog_all   = data.drop(columns=[cfg.TARGET])
exog_train = exog_all.loc[y_train.index]
exog_test  = exog_all.loc[y_test.index]

print(f"Train: {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)} obs)")
print(f"Test:  {y_test.index.min()} -> {y_test.index.max()}  ({len(y_test)} obs)")

In [ ]:
assessment = assess_stationarity(y_train, "Appliances (train)")

print("=== ADF ===")
for key in ["statistic", "p_value", "used_lag", "n_observations", "reject_null_5pct", "conclusion"]:
    print(f"  {key:20s} {assessment['adf'][key]}")

print("\n=== KPSS ===")
for key in ["statistic", "p_value", "lags", "reject_null_5pct", "conclusion"]:
    print(f"  {key:20s} {assessment['kpss'][key]}")

print("\n=== VERDICT ===")
print("  ", assessment["verdict"])
print("  ", assessment["recommended_action"])
print("  differenced tests:", assessment["differenced_tests"])

In [ ]:
evidence   = seasonality_evidence(y_train)
strengths  = component_strengths(y_train)
period_doc = seasonal_period_justification(y_train)

print("Seasonal strength Fs:", strengths["seasonal_strength_Fs"])
print("Trend strength Ft:   ", strengths["trend_strength_Ft"])
print("95% band:            ", round(evidence["white_noise_band_95pct"], 4))
print("\nACF at multiples of 24h:", evidence["acf_at_daily_multiples"])
print("ACF at multiples of 168h:", evidence["acf_at_weekly_multiples"])
print("\nLargest ACF: lag", evidence["largest_acf_lag"], "=", evidence["largest_acf_value"])
print("\n", period_doc["note"])

In [ ]:
display(Image(filename=str(plot_acf_pacf(y_train))))
display(Image(filename=str(plot_decomposition(decompose(y_train)))))
display(Image(filename=str(plot_stationarity_diagnostics(y_train))))

## D. Forecasting problem definition

**Part 3.** The 24-hour horizon and the 14-day test period are reconciled by a
rolling-origin design: 14 origins spaced 24 hours apart, each producing a
24-step forecast, pooled into one comparable sample of 336 points.

In [ ]:
experiment = {
    "target": cfg.TARGET,
    "units": cfg.TARGET_UNITS,
    "frequency": "hourly (resampled from 10-minute observations)",
    "train_start": str(y_train.index.min()), "train_end": str(y_train.index.max()),
    "n_train": int(len(y_train)),
    "test_start": str(y_test.index.min()), "test_end": str(y_test.index.max()),
    "n_test": int(len(y_test)), "test_days": cfg.TEST_DAYS,
    "horizon": cfg.HORIZON, "n_origins": cfg.N_ORIGINS,
    "split_type": "chronological hold-out; no random splitting",
    "evaluation_protocol": (
        f"Rolling origin: {cfg.N_ORIGINS} origins spaced {cfg.HORIZON} h apart, "
        f"each a {cfg.HORIZON}-step-ahead forecast, pooled over {cfg.TEST_STEPS} points."),
    "metrics": ["MAE", "RMSE", "MASE", "Bias", "sMAPE"],
    "primary_metric": cfg.PRIMARY_METRIC,
    "mape_excluded_reason": (
        "Target is strictly positive so MAPE is computable, but it is dominated by "
        "low night-time values and penalises over-forecasting asymmetrically."),
}
for key, value in experiment.items():
    print(f"{key:22s} {value}")

forecasts, extras, runtimes = {}, {}, {}

## E. Benchmark models

**Part 4.**

In [ ]:
t0 = time.time()
bench, meta = rolling_origin_benchmarks(y, y_test.index)
forecasts.update(bench)
runtimes["benchmarks"] = round(time.time() - t0, 2)

build_comparison_table(bench, y_test, y_train).round(3)

In [ ]:
display(Image(filename=str(plot_benchmark_forecasts(y_train, y_test, bench))))

## F. SARIMA / SARIMAX

**Part 5.** The grid search covers the complete required space
`p ∈ [0,6] × d ∈ [0,2] × q ∈ [0,6]` (147 models), then crosses the best
non-seasonal orders with the seasonal grid at `s = 24`.

Set `RUN_FULL_GRID = True` to refit everything (~35 minutes). Left `False`, the
selected order from the completed search is used and the two final models are
still fitted here from scratch.

In [ ]:
RUN_FULL_GRID = False   # True -> refit all 168 candidates (~35 min)

t0 = time.time()
if RUN_FULL_GRID:
    outcome        = run_full_grid_search(y_train)
    order          = outcome["best_order"]
    seasonal_order = outcome["best_seasonal_order"]
    grid           = outcome["results"]
    best_aic, best_bic = outcome["best_aic"], outcome["best_bic"]
else:
    # Result of the completed search; the models below are still fitted here.
    order, seasonal_order = (5, 0, 3), (0, 1, 1, 24)
    grid, best_aic, best_bic = None, None, None
    print("Using the order selected by the completed grid search.")
    print("Set RUN_FULL_GRID = True to reproduce the search yourself.")

print("Selected order:", order, seasonal_order)
runtimes["grid_search"] = round(time.time() - t0, 2)

In [ ]:
if grid is not None:
    display(grid[grid["status"] == "ok"].sort_values("aic").head(10)[
        ["order", "seasonal_order", "aic", "bic", "n_params", "fit_seconds"]])
    print(grid["status"].value_counts().to_dict())

### Exogenous variable selection

Two screens, both computed on the **training period only**: relevance
(|correlation|) and redundancy (variance inflation factor).

In [ ]:
selection = select_exog(data, y_train)
print("Candidates:      ", selection["candidates"])
print("Correlations:    ", selection["correlations_with_target"])
print("Dropped (low r): ", selection["dropped_low_correlation"])
print("Dropped (VIF):   ", selection["dropped_collinear"])
print("SELECTED:        ", selection["selected"])

### Fit both variants

A fitted state-space SARIMAX holds well over 1 GB once its smoother output is
materialised, so the exogenous variant is fitted first, its AIC recorded, and
the object released before the target-only model is built.

In [ ]:
t0 = time.time()
selected = selection["selected"]

fit_exog = fit_final_sarimax(y_train, order, seasonal_order,
                             exog_train=exog_train[selected])
aic_exog = float(fit_exog.aic)
intervals_exog = rolling_origin_forecast(fit_exog, y_test,
                                         exog_test=exog_test[selected])
forecasts["sarimax_exog_conditional"] = intervals_exog["forecast"]
extras["sarimax_exog_conditional"] = {"forecast_type": "conditional"}

del fit_exog          # release ~1.3 GB before the next fit
gc.collect()
print("AIC with exog:", round(aic_exog, 2))

In [ ]:
fit_plain = fit_final_sarimax(y_train, order, seasonal_order)
intervals_plain = rolling_origin_forecast(fit_plain, y_test)
forecasts["sarima"] = intervals_plain["forecast"]

print(f"AIC target-only : {fit_plain.aic:.2f}")
print(f"AIC with exog   : {aic_exog:.2f}")
print("\nLower AIC is better - does adding weather actually help in-sample?")
runtimes["sarimax"] = round(time.time() - t0, 2)

### Residual diagnostics

**Ljung-Box** — leftover autocorrelation. **Jarque-Bera** — normality, which
governs whether the analytic prediction intervals are trustworthy.
**Heteroskedasticity** — stability of the residual variance.

In [ ]:
diagnostics = residual_diagnostics(fit_plain)
for key, value in diagnostics.items():
    if key != "ljung_box":
        print(f"{key:38s} {value}")
print()
display(pd.DataFrame(diagnostics["ljung_box"]))

In [ ]:
resid = pd.Series(fit_plain.resid).iloc[cfg.SEASONAL_PERIOD:]
display(Image(filename=str(plot_residual_diagnostics(resid))))
display(Image(filename=str(plot_forecast_with_intervals(
    y_train, y_test, intervals_plain, f"SARIMA{order}{seasonal_order}", "12_sarima_forecast"))))

In [ ]:
coverage = ((y_test >= intervals_plain["lower"]) & (y_test <= intervals_plain["upper"])).mean()
print(f"Nominal coverage:   95.0%")
print(f"Empirical coverage: {coverage:.1%}")
print("\nA shortfall indicates the Gaussian interval assumption is violated.")

intervals_plain.to_csv(cfg.FORECAST_DIR / "sarima_intervals.csv")
intervals_exog.to_csv(cfg.FORECAST_DIR / "sarimax_exog_intervals.csv")
del fit_plain; gc.collect()

## G. Feature engineering and the ML model

**Parts 6–7.** The leakage self-test perturbs a single target observation and
asserts the corresponding feature row is unchanged.

In [ ]:
selftest = leakage_self_test(y_train.iloc[-1500:], exog_train.iloc[-1500:])
for regime, record in selftest.items():
    print(f"{regime:14s} leak={record['contemporaneous_target_leak']}  "
          f"columns={record['leaking_columns']}  features={record['n_features']}")

In [ ]:
table_op = make_feature_frame(y_train, exog_train, regime="operational")
for group, columns in feature_groups(table_op.columns).items():
    print(f"  {group:14s} {len(columns):3d}   e.g. {columns[:3]}")

In [ ]:
t0 = time.time()
tuning = select_hyperparameters(table_op, y_train.loc[table_op.index])
print("Best params:", tuning["best_params"])
print("CV RMSE:    ", round(tuning["best_cv_rmse"], 3))
display(tuning["cv_table"])

### Recursive multi-step forecasting

The critical step. Predicting the test rows directly from a full feature table
would use the **observed** future target through `lag_1 … lag_12`, which does
not exist at a 24-hour origin. Instead the working series is extended one hour
at a time with the model's own predictions.

In [ ]:
observed = attach_observed_test(y_train, y_test)

fitted_op = fit_ml_model(y_train, exog_train, "operational", tuning["best_params"])
forecasts["ml_operational"] = recursive_forecast(fitted_op, observed, y_test.index, exog_all)

fitted_cond = fit_ml_model(y_train, exog_train, "conditional", tuning["best_params"])
forecasts["ml_conditional"] = recursive_forecast(fitted_cond, observed, y_test.index, exog_all)
extras["ml_conditional"] = {"forecast_type": "conditional"}

runtimes["ml"] = round(time.time() - t0, 2)
build_comparison_table(
    {k: forecasts[k] for k in ["ml_operational", "ml_conditional"]}, y_test, y_train
).round(3)

### Interpretation

Permutation importance is measured on a held-out tail of the **training**
period, never on the test set.

In [ ]:
importance = compute_permutation_importance(fitted_op, y_train, exog_train)
display(importance.head(15))
display(aggregate_importance_by_group(importance))
display(Image(filename=str(plot_feature_importance(importance))))

### Feature-group ablation

Each stage adds one group and is evaluated with the **same** recursive
rolling-origin protocol, so differences are attributable to the features.

In [ ]:
ablation = run_feature_ablation(
    y_train=y_train, y_test=y_test, exog_train=exog_train, exog_full=exog_all,
    regime="operational", params=tuning["best_params"],
    mase_scale_value=mase_scale(y_train),
)
display(ablation.round(3))
display(Image(filename=str(plot_feature_ablation(ablation))))

ablation.to_csv(cfg.METRICS_DIR / "ml_feature_ablation.csv", index=False)
importance.to_csv(cfg.METRICS_DIR / "ml_feature_importance.csv", index=False)

> Compare the ablation against the permutation importance. Importance is
> measured **one step ahead**; the ablation is **24 steps ahead** with
> predictions feeding back. Any disagreement between them is a finding, not an
> error.

## H. Foundation model

**Part 8.** Runs genuine Chronos or reports honestly that it could not. No
fallback forecast is ever produced.

In [ ]:
t0 = time.time()
foundation = run_foundation_model(observed, y_test.index)
print("STATUS:", foundation["status"], "|", foundation["result_marker"])

if foundation["status"] == "executed":
    forecasts["foundation_chronos"] = foundation["forecast"]
    print("Runtime:", foundation["runtime_seconds"], "s")
    print("Mode:   ", foundation["mode"])
    foundation["intervals"].to_csv(cfg.FORECAST_DIR / "chronos_intervals.csv")
    display(foundation["intervals"].head())
else:
    print("REASON:", foundation["reason"])
    print("\nNo forecast produced, so no row enters the comparison table.")
    print("Assignment Question 4 cannot be answered until this runs.")
    for step, text in foundation["how_to_run"].items():
        print(f"  {step}: {text}")

runtimes["foundation"] = round(time.time() - t0, 2)

In [ ]:
if foundation["status"] == "executed":
    display(Image(filename=str(plot_forecast_with_intervals(
        y_train, y_test, foundation["intervals"], "Chronos (zero-shot)",
        "14_foundation_forecast", interval_label="80% quantile interval (q10-q90)"))))

## I. Evaluation

**Parts 9–10.** Every model on the identical 336-point sample.

In [ ]:
comparison = build_comparison_table(forecasts, y_test, y_train, extras=extras)
if "forecast_type" not in comparison.columns:
    comparison["forecast_type"] = "operational"
comparison["forecast_type"] = comparison["forecast_type"].fillna("operational")

comparison.to_csv(cfg.METRICS_DIR / "model_comparison.csv", index=False)
comparison.round(3)

### Statistical significance

A metric gap is not evidence on its own. Diebold-Mariano asks whether the
difference exceeds sampling noise. **Negative** statistic favours the model;
`significant_at_5pct` must be `True` to claim a real improvement.

In [ ]:
best_benchmark = strongest_benchmark(comparison)
print("Strongest benchmark by", cfg.PRIMARY_METRIC, "->", best_benchmark, "\n")

dm = pairwise_dm_against(y_test, forecasts, reference=best_benchmark)
dm.to_csv(cfg.METRICS_DIR / "diebold_mariano_tests.csv", index=False)
dm

In [ ]:
reference_rmse = float(comparison.loc[comparison["model"] == best_benchmark, "RMSE"].iloc[0])
skill = {str(r["model"]): round(skill_score(reference_rmse, float(r["RMSE"])), 2)
         for _, r in comparison.iterrows()}
print(f"RMSE skill vs {best_benchmark} (%, positive = better):")
for name, value in skill.items():
    print(f"  {name:28s} {value:+.2f}")

In [ ]:
horizon_errors = errors_by_horizon_step(forecasts, y_test, meta["step"])
horizon_errors.to_csv(cfg.METRICS_DIR / "error_by_horizon_step.csv", index=False)
display(horizon_errors.pivot(index="step", columns="model", values="RMSE").iloc[[0, 5, 11, 17, 23]].round(2))

display(Image(filename=str(plot_forecast_comparison(y_train, y_test, forecasts))))
display(Image(filename=str(plot_error_comparison(comparison))))
display(Image(filename=str(plot_error_by_horizon(horizon_errors))))

In [ ]:
forecast_frame = pd.DataFrame({"actual": y_test})
for name, series in forecasts.items():
    forecast_frame[name] = series.reindex(y_test.index)
forecast_frame = forecast_frame.join(meta)
forecast_frame.to_csv(cfg.FORECAST_DIR / "all_forecasts.csv")
forecast_frame.head()

### Operational versus conditional

Only the **operational** models could be deployed. The conditional ones assume
future weather and sensor readings are known exactly.

In [ ]:
for kind, group in comparison.groupby("forecast_type"):
    print(f"--- {kind} ---")
    print(group[["rank", "model", "MASE", "RMSE"]].to_string(index=False), "\n")

## J. Analysis outputs and leakage audit

**Parts 11–12.** Facts and questions; no interpretation.

In [ ]:
runtimes["total"] = round(sum(v for k, v in runtimes.items() if k != "total"), 2)

summary = build_analysis_summary(
    quality=quality.to_dict(),
    aggregation={**aggregation, "interpolation_note": check_interpolation_safety(aggregation)},
    eda=eda_record,
    stationarity={"target_series": assessment, "seasonality_evidence": evidence,
                  "component_strengths": strengths,
                  "seasonal_period_justification": period_doc},
    experiment=experiment,
    sarimax={"selected_order": str(order), "selected_seasonal_order": str(seasonal_order),
             "selected_aic": best_aic, "selected_bic": best_bic,
             "final_fit_aic_with_exog": aic_exog,
             "n_models_attempted": None if grid is None else int(len(grid)),
             "n_models_converged": None if grid is None else int((grid["status"] == "ok").sum()),
             "n_models_failed": None if grid is None else int((grid["status"] != "ok").sum()),
             "grid_definition": {"p": cfg.SEARCH_P, "d": cfg.SEARCH_D, "q": cfg.SEARCH_Q,
                                 "P": cfg.SEARCH_SEASONAL_P, "D": cfg.SEARCH_SEASONAL_D,
                                 "Q": cfg.SEARCH_SEASONAL_Q, "s": cfg.SEASONAL_PERIOD},
             "search_seconds": runtimes.get("grid_search"),
             "exog_selection": selection, "selected_exog": selected,
             "residual_diagnostics": diagnostics},
    ml={"backend": fitted_op["backend"], "best_params": tuning["best_params"],
        "cv_rmse": tuning["best_cv_rmse"], "n_train_rows": fitted_op["n_train_rows"],
        "leakage_self_test": selftest,
        "feature_importance_top20": importance.head(20).to_dict(orient="records"),
        "feature_importance_by_group": aggregate_importance_by_group(importance).to_dict(orient="records"),
        "ablation": ablation.to_dict(orient="records")},
    foundation={k: v for k, v in foundation.items() if k not in ("forecast", "intervals")},
    comparison=comparison, dm_tests=dm, runtimes=runtimes,
)
summary["strongest_benchmark"] = best_benchmark
summary["skill_scores_rmse_vs_best_benchmark"] = skill

json_path, md_path = write_analysis_summary(summary)
print("Wrote:", json_path)
print("Wrote:", md_path)

In [ ]:
audit = build_leakage_audit(
    aggregation=aggregation,
    interpolation_note=check_interpolation_safety(aggregation),
    feature_selftest=selftest,
    experiment=experiment,
    exog_selection=selection,
    ml_regimes={"conditional_included": "ml_conditional" in forecasts},
    foundation=foundation,
)
audit_path = write_leakage_audit(audit)
print("Wrote:", audit_path, "\n")
print(audit[:3000])

## K. Analytical questions

Prompts for the report, mapped to the six assignment questions. These are
questions, not answers — the interpretation is the assessed work.

In [ ]:
for section, questions in summary["analytical_questions"].items():
    print(f"\n{'=' * 70}\n{section}\n{'=' * 70}")
    for i, question in enumerate(questions, start=1):
        print(f"{i}. {question}")

## L. Download the results (Colab)

Zips every output so you can pull them off the Colab VM in one click.

In [ ]:
import shutil
archive = shutil.make_archive("appliance_forecasting_outputs", "zip", str(cfg.OUTPUT_DIR))
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab - the archive is in the working directory.")

---

## Summary of what ran

| Part | Section |
|---|---|
| 1 Data + EDA | A, B |
| 2 Stationarity | C |
| 3 Problem definition | D |
| 4 Benchmarks | E |
| 5 SARIMAX | F |
| 6–7 Features + ML | G |
| 8 Foundation model | H |
| 9–10 Evaluation + figures | I |
| 11–12 Analysis + leakage audit | J, K |

**Reminder on honesty:** if the foundation model reports `not_executed`, no
claim about its accuracy is supported by this run. Report that plainly rather
than substituting a benchmark — the original demo pipeline this project
replaced did exactly that, and it would be a serious error in a submitted
report.